# Planner-Director Architecture â€” Swiss Legal Citation Retrieval

**Architecture v2.0** â€” Replaces HyDE-based flat ReAct agent with structured Planner-Director pipeline.

Key changes from `03_hyde_kaggle.ipynb`:
- **Planner LLM** decomposes question â†’ 3-6 research directions (GBNF-constrained)
- **Direction Executors** search sequentially with metadata filters + per-code taxonomy
- **No HyDE** â€” routing guides + German queries from planner replace it
- **Metadata-filtered FAISS** â€” searches targeted subsets (20x smaller search space)
- **~27s/question** vs ~4min (9x faster)

Requires: Kaggle GPU T4 x2, Internet ON for model downloads.

In [1]:
# Cell 2: Install dependencies
!pip install -q sentence-transformers faiss-cpu pydantic transformers rank-bm25 numpy
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.1 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 621.3 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


In [ ]:
# Cell 3: Imports, Paths & Configuration
import os, sys, re, gc, time, pickle, shutil, subprocess, json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# ---- Paths (Kaggle only) ----
DATA_PATH = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")
MODEL_PATH = Path("/kaggle/input/datasets/charan1996/mistral-7b-gguf")
CHECKPOINT_DS = Path("/kaggle/input/datasets/charan1996/rag-checkpoints-v2")  # persistent read
OUTPUT_PATH = Path("/kaggle/working")
INDEX_PATH = Path("/kaggle/working/cache")

# ---- Corpus CSV paths ----
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"
TRAIN_CSV = DATA_PATH / "train.csv"
TEST_CSV = DATA_PATH / "test.csv"

# ---- Index cache paths ----
CORPUS_CACHE_PATH = INDEX_PATH / "corpus_documents.pkl"
FAISS_LAWS_PATH = INDEX_PATH / "faiss_laws_qwen3_embeddings.pkl"
FAISS_COURTS_PATH = INDEX_PATH / "faiss_courts_qwen3_embeddings.pkl"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

# ---- Configuration ----
CONFIG = {
    # --- LLM (Mistral-7B on GPU 0) ---
    "model_file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    "n_ctx": 16384,
    "n_threads": 8,
    "n_gpu_layers": -1,
    "max_tokens_planner": 6000,
    "max_tokens_executor": 800,
    "temperature": 0.1,
    "temperature_executor": 0.3,
    # --- Embedding (Qwen3-Embedding on GPU 1) ---
    "embed_model": "Qwen/Qwen3-Embedding-0.6B",
    "embed_dim": 1024,
    "embed_max_length": 1024,
    "embed_batch_size": 8,
    "prompt_doc_law": "Instruct: Represent this Swiss federal statute article in German for legal citation retrieval\nDocument: ",
    "prompt_doc_court": "Instruct: Represent this Swiss Federal Court decision excerpt in German for legal citation retrieval\nDocument: ",
    "prompt_query_law": "Instruct: Given German legal search terms, retrieve relevant Swiss federal statute articles from the SR collection\nQuery: ",
    "prompt_query_court": "Instruct: Given German legal search terms, retrieve relevant Swiss Federal Court decisions (BGE)\nQuery: ",
    # --- Reranker (Qwen3-Reranker on GPU 1) ---
    "rerank_model": "Qwen/Qwen3-Reranker-0.6B",
    "rerank_batch_size": 8,
    "rerank_score_cutoff": 0.0,  # DISABLED - reranker bypassed, using RRF scores directly
    "max_final_citations": 60,
    # --- Pipeline ---
    "max_executor_iterations": 3,
    "executor_timeout_sec": 15,
    "adaptive_fallback_threshold": 5,
    "rrf_k": 60,
    "search_top_k": 10,
    "courts_sample_n": 200_000,
}

# ---- Validation ----
assert LAWS_CSV.exists(), f"Laws CSV not found: {LAWS_CSV}"
assert COURTS_CSV.exists(), f"Courts CSV not found: {COURTS_CSV}"
assert TEST_CSV.exists(), f"Test CSV not found: {TEST_CSV}"

print(f"Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)")
print(f"Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)")
print(f"Test CSV: {TEST_CSV}")

# ---- Restore from persistent dataset (if available from previous run) ----
if CHECKPOINT_DS.exists():
    restored = 0
    for f in CHECKPOINT_DS.iterdir():
        if f.suffix == '.pkl':
            dest = INDEX_PATH / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                restored += 1
        elif f.suffix in ('.txt', '.gbnf'):
            # Also copy prompt/grammar files to OUTPUT_PATH for push reliability
            dest = OUTPUT_PATH / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                restored += 1
    if restored:
        print(f"â™»ï¸  Restored {restored} cached files from {CHECKPOINT_DS}")
    else:
        print(f"âœ… Checkpoint dataset found â€” caches already in place")
else:
    print(f"No checkpoint dataset yet â€” will build from scratch")

# ---- Save helpers ----
DATASET_SLUG = "charan1996/rag-checkpoints-v2"
_DATASET_STAGING = OUTPUT_PATH / "_dataset_staging"
_push_count = 0

def _pull_existing_dataset():
    """Download current dataset version into staging so we don't lose old files."""
    tmp_dl = OUTPUT_PATH / "_dataset_download"
    if tmp_dl.exists():
        shutil.rmtree(tmp_dl)
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", DATASET_SLUG,
         "-p", str(tmp_dl), "--unzip"],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode == 0 and tmp_dl.exists():
        for f in tmp_dl.iterdir():
            if f.name != "dataset-metadata.json":
                shutil.copy2(f, _DATASET_STAGING / f.name)
        shutil.rmtree(tmp_dl)
        print(f"   â™»ï¸  Pulled {sum(1 for _ in _DATASET_STAGING.iterdir())} existing files from dataset")
    # If download fails (first push, no dataset yet), that's fine â€” nothing to preserve


def _push_to_dataset():
    """Push ALL checkpoint + prompt files to Kaggle Dataset.
    Includes prompt/grammar files so no version ever loses them.
    RAISES RuntimeError on failure â€” notebook STOPS so you know data is NOT saved."""
    global _push_count
    # --- Core pipeline outputs ---
    all_paths = [
        FAISS_LAWS_PATH, FAISS_COURTS_PATH, CORPUS_CACHE_PATH,
        OUTPUT_PATH / "predictions_checkpoint.pkl",
        OUTPUT_PATH / "val_predictions_checkpoint.pkl",
        OUTPUT_PATH / "submission.csv",
        OUTPUT_PATH / "submission_progress.csv",
        OUTPUT_PATH / "pipeline_debug_log.txt",
    ]
    # --- Prompt/grammar files (from checkpoint dataset or local) ---
    _PROMPT_GRAMMAR_FILES = [
        "planner_system.txt", "executor_system.txt", "executor_procedural.txt",
        "planner.gbnf", "executor.gbnf", "fallback_rules.txt",
        "swiss_legal_system.txt", "routing_guide_laws.txt", "routing_guide_courts.txt",
        "terminology_bridge.txt", "procedural_defaults.txt",
    ]
    for fname in _PROMPT_GRAMMAR_FILES:
        # Check CHECKPOINT_DS (read-only input), then OUTPUT_PATH
        for search_dir in [CHECKPOINT_DS, OUTPUT_PATH]:
            candidate = search_dir / fname
            if candidate.exists():
                all_paths.append(candidate)
                break
    existing_files = [Path(p) for p in all_paths if Path(p).exists()]
    if not existing_files:
        print("âš ï¸  No files to push yet")
        return
    if _DATASET_STAGING.exists():
        shutil.rmtree(_DATASET_STAGING)
    _DATASET_STAGING.mkdir(parents=True, exist_ok=True)
    # Copy all files to staging (no need to re-download â€” we have everything locally)
    for fpath in existing_files:
        shutil.copy2(fpath, _DATASET_STAGING / fpath.name)
    metadata = {"id": DATASET_SLUG, "title": "rag-checkpoints-v2", "licenses": [{"name": "CC0-1.0"}]}
    with open(_DATASET_STAGING / "dataset-metadata.json", "w") as f:
        json.dump(metadata, f)
    _push_count += 1
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(_DATASET_STAGING),
         "-m", f"checkpoint #{_push_count}", "--dir-mode", "zip"],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode != 0 and ("404" in result.stderr or "not found" in result.stderr.lower()):
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(_DATASET_STAGING)],
            capture_output=True, text=True, timeout=600
        )
    if result.returncode != 0:
        msg = f"âŒ DATASET PUSH FAILED!\n  stdout: {result.stdout}\n  stderr: {result.stderr}"
        print(msg)
        raise RuntimeError(msg)
    file_list = ", ".join(f.name for f in existing_files)
    print(f"âœ… SAVED to kaggle.com/datasets/{DATASET_SLUG} (push #{_push_count}): {file_list}")


def _save_checkpoint(*files):
    """Copy files to /kaggle/working/ then push ALL checkpoints to permanent dataset."""
    for fpath in files:
        fpath = Path(fpath)
        if not fpath.exists():
            continue
        dest = OUTPUT_PATH / fpath.name
        if dest != fpath:
            shutil.copy2(fpath, dest)
    _push_to_dataset()


print(f"ðŸ’¾ Permanent save target: kaggle.com/datasets/{DATASET_SLUG}")
print(f"   If push fails â†’ notebook STOPS (no silent data loss)")

Laws CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv (73.0 MB)
Courts CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv (2.43 GB)
Test CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv
âœ… Checkpoint dataset found â€” caches already in place
ðŸ’¾ Permanent save target: kaggle.com/datasets/charan1996/rag-checkpoints-v2
   If push fails â†’ notebook STOPS (no silent data loss)


In [3]:
# Cell 4: Load Corpus (Laws + Courts)

def load_csv_corpus(csv_path: Path) -> list[dict]:
    """Load full corpus from CSV."""
    docs = []
    for chunk in pd.read_csv(csv_path, chunksize=50_000):
        for _, row in chunk.iterrows():
            docs.append({"citation": str(row["citation"]), "text": str(row.get("text", ""))})
    return docs


def load_csv_corpus_sampled(csv_path: Path, sample_n: int, seed: int = 42) -> list[dict]:
    """Load sampled corpus â€” memory-efficient for courts (2.4M docs)."""
    print(f"  Random sampling {sample_n:,} rows from {csv_path.name}...")
    df = pd.read_csv(csv_path, usecols=["citation", "text"], dtype={"citation": str, "text": str}, na_filter=False)
    total = len(df)
    if total <= sample_n:
        print(f"  Corpus has only {total:,} rows â€” using all")
    else:
        df = df.sample(n=sample_n, random_state=seed)
        print(f"  Sampled {sample_n:,} from {total:,} total")
    documents = [{"citation": r["citation"], "text": r["text"]} for _, r in df.iterrows()]
    return documents


# ---- Load Corpora ----
if CORPUS_CACHE_PATH.exists():
    print(f"Loading cached corpora from {CORPUS_CACHE_PATH}")
    with open(CORPUS_CACHE_PATH, 'rb') as f:
        _corpus_data = pickle.load(f)
    laws_documents = _corpus_data["laws"]
    courts_documents = _corpus_data["courts"]
    print(f"  Laws: {len(laws_documents):,}, Courts: {len(courts_documents):,}")
    del _corpus_data
else:
    laws_documents = load_csv_corpus(LAWS_CSV)
    print(f"Laws corpus: {len(laws_documents):,} documents")

    courts_documents = load_csv_corpus_sampled(COURTS_CSV, sample_n=CONFIG["courts_sample_n"])
    print(f"Courts corpus: {len(courts_documents):,} documents")

    # Cache for fast reload
    with open(CORPUS_CACHE_PATH, 'wb') as f:
        pickle.dump({"laws": laws_documents, "courts": courts_documents}, f)
    print(f"  Cached to {CORPUS_CACHE_PATH}")

_save_checkpoint(CORPUS_CACHE_PATH)

print(f"\nLaws:   {len(laws_documents):>10,} documents")
print(f"Courts: {len(courts_documents):>10,} documents")
print(f"Total:  {len(laws_documents) + len(courts_documents):>10,} documents")

Loading cached corpora from /kaggle/working/cache/corpus_documents.pkl
  Laws: 175,933, Courts: 200,000
âœ… SAVED to kaggle.com/datasets/charan1996/rag-checkpoints-v2 (push #1): faiss_laws_qwen3_embeddings.pkl, faiss_courts_qwen3_embeddings.pkl, corpus_documents.pkl, planner_system.txt, executor_system.txt, executor_procedural.txt, planner.gbnf, executor.gbnf, fallback_rules.txt, swiss_legal_system.txt, routing_guide_laws.txt, routing_guide_courts.txt, terminology_bridge.txt, procedural_defaults.txt

Laws:      175,933 documents
Courts:    200,000 documents
Total:     375,933 documents


In [43]:
# Cell 5: Build Metadata Filter Indices + Type Registry
# Citation format: "Art. 60 OR", "Art. 975 ZGB", "Art. 3 Abs. 1 131.211" (laws)
#                  "1B_123/2020", "BGE 137 IV 122 E. 6.2" (courts)
# The type CODE is the trailing abbreviation for laws, prefix/BGE pattern for courts.

_COURT_PREFIX_RE = re.compile(r"^(\d[A-Z]_)")
_BGE_RE = re.compile(r"^BGE\s+(\d+)\s+(I{1,3}V?|V)\s")
# Matches Swiss abbreviations with mixed case: StGB, StPO, SchKG, JStPO, AsylG, etc.
_LAW_TYPE_RE = re.compile(r'\b([A-Z][A-Za-z]*[A-Z][a-z]?)\s*$')
_LAW_TYPE_FALLBACK_RE = re.compile(r'\b([A-Z]{2,}[a-z]?)\b')


def get_law_type(citation: str) -> str:
    """Extract statute abbreviation from law citation.
    Handles both all-caps (OR, ZGB, BGG) and mixed-case (StGB, StPO, SchKG, JStPO).
    'Art. 60 OR' â†’ 'OR', 'Art. 1 StGB' â†’ 'StGB', 'Art. 3 Abs. 1 131.211' â†’ 'OTHER'
    """
    c = citation.strip()
    # Primary: trailing abbreviation with â‰¥2 uppercase letters (allows lowercase between)
    match = _LAW_TYPE_RE.search(c)
    if match:
        return match.group(1)
    # Fallback: last all-caps word
    matches = _LAW_TYPE_FALLBACK_RE.findall(c)
    return matches[-1] if matches else "OTHER"


def get_court_type(citation: str) -> str:
    """Extract court division code from court citation.
    '1B_123/2020' â†’ '1B_', 'BGE 137 IV 122' â†’ 'BGE_IV'
    Keeps prefix style to match routing_guide_courts.txt sections.
    """
    # Pattern 1: '1B_123/2020' â†’ '1B_'
    m = _COURT_PREFIX_RE.match(citation)
    if m:
        return m.group(1)
    # Pattern 2: 'BGE 137 IV 122' â†’ 'BGE_IV'
    m = _BGE_RE.match(citation)
    if m:
        return f"BGE_{m.group(2)}"
    return "OTHER"


# --- Build index mappings ---
print("Building metadata filter indices...")
t0 = time.time()

# Laws: type_code â†’ numpy array of doc indices
law_code_to_indices: dict[str, np.ndarray] = {}
_law_codes_raw = [get_law_type(d["citation"]) for d in laws_documents]
for idx, code in enumerate(_law_codes_raw):
    law_code_to_indices.setdefault(code, []).append(idx)
law_code_to_indices = {k: np.array(v, dtype=np.int64) for k, v in law_code_to_indices.items()}

# Courts: type_code â†’ numpy array of doc indices
court_code_to_indices: dict[str, np.ndarray] = {}
_court_codes_raw = [get_court_type(d["citation"]) for d in courts_documents]
for idx, code in enumerate(_court_codes_raw):
    court_code_to_indices.setdefault(code, []).append(idx)
court_code_to_indices = {k: np.array(v, dtype=np.int64) for k, v in court_code_to_indices.items()}

# Type counts for registry
law_type_counts = {k: len(v) for k, v in law_code_to_indices.items()}
court_type_counts = {k: len(v) for k, v in court_code_to_indices.items()}

# Filter: keep only codes with â‰¥10 docs (laws) or â‰¥50 docs (courts)
# Lower threshold than before â€” lets Planner use precise codes
available_law_codes = sorted(k for k, v in law_code_to_indices.items() if len(v) >= 10 and k != "OTHER")
available_court_codes = sorted(k for k, v in court_code_to_indices.items() if len(v) >= 50 and k != "OTHER")

# Frequency-annotated strings for Planner context (like a lawyer's reference card)
# Top-40 law codes only (saves ~7800 chars vs full 200+ list)
LAW_TYPES_FOR_PROMPT = ", ".join(
    f"{t}({c})" for t, c in sorted(
        [(k, v) for k, v in law_type_counts.items() if k != "OTHER" and v >= 10],
        key=lambda x: -x[1]
    )[:40]
)
COURT_TYPES_FOR_PROMPT = ", ".join(
    f"{t}({c})" for t, c in sorted(court_type_counts.items(), key=lambda x: -x[1])
    if t != "OTHER" and c >= 50
)

# Citation set (for validating procedural defaults exist in corpus)
corpus_citation_set = set(d["citation"] for d in laws_documents) | set(d["citation"] for d in courts_documents)

# Citation â†’ document text lookup (for reranker â€” uses first 1500 chars)
# If duplicate citations exist, keeps the longest text
citation_to_text: dict[str, str] = {}
for _doc in laws_documents:
    _cit, _txt = _doc["citation"], _doc["text"][:1500]
    if _cit not in citation_to_text or len(_txt) > len(citation_to_text[_cit]):
        citation_to_text[_cit] = _txt
for _doc in courts_documents:
    _cit, _txt = _doc["citation"], _doc["text"][:1500]
    if _cit not in citation_to_text or len(_txt) > len(citation_to_text[_cit]):
        citation_to_text[_cit] = _txt

print(f"  Type registry: {len(law_type_counts)} law types, {len(court_type_counts)} court types")
print(f"  Laws:   {len(available_law_codes)} available codes (from {len(law_code_to_indices)} total, threshold â‰¥10)")
print(f"  Courts: {len(available_court_codes)} available codes (from {len(court_code_to_indices)} total, threshold â‰¥50)")
print(f"  Built in {time.time()-t0:.1f}s")
print(f"  Top 15 law codes: {[f'{k}({v})' for k, v in sorted(law_type_counts.items(), key=lambda x: -x[1])[:15]]}")
print(f"  Top 10 court codes: {[f'{k}({v})' for k, v in sorted(court_type_counts.items(), key=lambda x: -x[1])[:10]]}")
print(f"  Citationâ†’text lookup: {len(citation_to_text):,} entries (for reranker)")


Building metadata filter indices...
  Type registry: 929 law types, 42 court types
  Laws:   920 available codes (from 929 total, threshold â‰¥10)
  Courts: 29 available codes (from 42 total, threshold â‰¥50)
  Built in 0.8s
  Top 15 law codes: ['OTHER(47403)', 'OR(3662)', 'FINMA(2589)', 'ZGB(2395)', 'StPO(1306)', 'StGB(1239)', 'ETH(1127)', 'TSV(1094)', 'SchKG(1060)', 'VTS(1022)', 'ZPO(857)', 'MStG(811)', 'ZV(717)', 'AVO(716)', 'TSchV(710)']
  Top 10 court codes: ['OTHER(35156)', '6B_(25579)', '2C_(25496)', '5A_(19745)', '8C_(18625)', '9C_(16790)', '4A_(15351)', '1C_(14959)', '1B_(8245)', '7B_(4105)']
  Citationâ†’text lookup: 372,372 entries (for reranker)


In [27]:
# Cell 6: Torch setup (needed by embedding model + reranker on GPU 1)
import torch
torch.cuda.set_device(0)  # Default GPU for LLM (loaded later after index building)
print(f"PyTorch ready: {torch.cuda.device_count()} GPU(s) available")

PyTorch ready: 2 GPU(s) available


In [28]:
# Cell 7: Load Embedding Model (GPU 1)
# NOTE: Reranker is loaded AFTER Cell 8 (embeddings) to avoid GPU 1 memory pressure.
# Reranker occupies ~1.2GB VRAM but isn't needed until aggregate_and_output() in Cell 15.
# Loading it here would starve the embedding model during the 175K+200K doc encoding.
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
_st_model = SentenceTransformer(
    CONFIG["embed_model"],
    device="cuda:1",
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.float16},
)
_st_model.max_seq_length = CONFIG["embed_max_length"]
print(f"  Embedding: {CONFIG['embed_model']} on cuda:1, dim={_st_model.get_sentence_embedding_dimension()}")
print(f"  âš  Reranker deferred â†’ will load after FAISS indices are built (Cell 8.5)")

Loading embedding model...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Embedding: Qwen/Qwen3-Embedding-0.6B on cuda:1, dim=1024
  âš  Reranker deferred â†’ will load after FAISS indices are built (Cell 8.5)


In [29]:
# Cell 8: Build FAISS Indices
# MEMORY OPTIMIZATION: Embed â†’ build FAISS â†’ del embeddings for EACH corpus independently.
# This avoids holding both embedding arrays (~1.46GB total) in RAM simultaneously.
# Peak RAM reduced from ~1.46GB to ~780MB during this cell.
import faiss


def embed_corpus(documents: list[dict], doc_type: str = "law", batch_size: int = 8) -> np.ndarray:
    """Embed corpus documents using instruction-prefixed encoding."""
    texts = [f"{d['citation']}: {d['text'][:1500]}" for d in documents]
    prompt = CONFIG["prompt_doc_law"] if doc_type == "law" else CONFIG["prompt_doc_court"]
    print(f"  Embedding {len(texts):,} docs, batch_size={batch_size}")
    t0 = time.time()
    embeddings = _st_model.encode(
        texts,
        prompt=prompt,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=batch_size,
    )
    print(f"  Done in {time.time()-t0:.1f}s ({len(texts)/(time.time()-t0):.0f} docs/sec)")
    return embeddings.astype("float32")


# â”€â”€â”€ Laws: Embed â†’ FAISS â†’ free immediately â”€â”€â”€
if FAISS_LAWS_PATH.exists():
    print(f"Loading cached law embeddings (Qwen3)...")
    with open(FAISS_LAWS_PATH, 'rb') as f:
        law_embeddings = pickle.load(f)
else:
    law_embeddings = embed_corpus(laws_documents, "law", CONFIG["embed_batch_size"])
    with open(FAISS_LAWS_PATH, 'wb') as f:
        pickle.dump(law_embeddings, f)
    _save_checkpoint(FAISS_LAWS_PATH)

dim = law_embeddings.shape[1]
faiss_law_index = faiss.IndexFlatIP(dim)
faiss_law_index.add(law_embeddings)
print(f"  Laws FAISS: {faiss_law_index.ntotal:,} vectors added")

# FREE law embeddings BEFORE starting courts (~680MB freed)
del law_embeddings
gc.collect()
torch.cuda.empty_cache()  # Release PyTorch's CUDA allocator blocks back to driver
print(f"  âœ“ Law embeddings freed â€” starting courts with clean RAM")

# â”€â”€â”€ Courts: Embed â†’ FAISS â†’ free immediately â”€â”€â”€
if FAISS_COURTS_PATH.exists():
    print(f"Loading cached court embeddings (Qwen3)...")
    with open(FAISS_COURTS_PATH, 'rb') as f:
        court_embeddings = pickle.load(f)
else:
    court_embeddings = embed_corpus(courts_documents, "court", CONFIG["embed_batch_size"])
    with open(FAISS_COURTS_PATH, 'wb') as f:
        pickle.dump(court_embeddings, f)
    _save_checkpoint(FAISS_COURTS_PATH)

faiss_court_index = faiss.IndexFlatIP(dim)
faiss_court_index.add(court_embeddings)
print(f"  Courts FAISS: {faiss_court_index.ntotal:,} vectors added")

# FREE court embeddings (~780MB freed)
del court_embeddings
gc.collect()
torch.cuda.empty_cache()  # Release PyTorch's CUDA allocator blocks back to driver

print(f"\nFAISS indices built:")
print(f"  Laws:   {faiss_law_index.ntotal:,} vectors, dim={dim}")
print(f"  Courts: {faiss_court_index.ntotal:,} vectors, dim={dim}")
_save_checkpoint(FAISS_LAWS_PATH, FAISS_COURTS_PATH)

Loading cached law embeddings (Qwen3)...
  Laws FAISS: 175,933 vectors added
  âœ“ Law embeddings freed â€” starting courts with clean RAM
Loading cached court embeddings (Qwen3)...
  Courts FAISS: 200,000 vectors added

FAISS indices built:
  Laws:   175,933 vectors, dim=1024
  Courts: 200,000 vectors, dim=1024
âœ… SAVED to kaggle.com/datasets/charan1996/rag-checkpoints-v2 (push #2): faiss_laws_qwen3_embeddings.pkl, faiss_courts_qwen3_embeddings.pkl, corpus_documents.pkl, pipeline_debug_log.txt, planner_system.txt, executor_system.txt, executor_procedural.txt, planner.gbnf, executor.gbnf, fallback_rules.txt, swiss_legal_system.txt, routing_guide_laws.txt, routing_guide_courts.txt, terminology_bridge.txt, procedural_defaults.txt


In [30]:
# Cell 8.5: Load Reranker (GPU 1) â€” AFTER embeddings freed
# Reranker is only needed in aggregate_and_output() (Cell 15).
# Loading it here (after raw embeddings are del'd) prevents GPU 1 VRAM starvation
# during the 375K-doc encoding in Cell 8.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F


class Qwen3Reranker:
    """Generative reranker using P(yes) vs P(no) logit scoring."""

    def __init__(self, model_name: str, device: str = "cuda:1", dtype=torch.float16):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name, padding_side="left", trust_remote_code=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype, trust_remote_code=True
        ).to(device).eval()
        self.device = device
        # Robust token ID resolution for yes/no scoring
        self.yes_id = self._resolve_token_id("Yes", ["yes", "YES", "true", "True"])
        self.no_id = self._resolve_token_id("No", ["no", "NO", "false", "False"])
        print(f"[Qwen3Reranker] yes_id={self.yes_id}, no_id={self.no_id}")

    def _resolve_token_id(self, primary: str, fallbacks: list) -> int:
        """Resolve a token string to its ID, trying multiple strategies."""
        unk = self.tokenizer.unk_token_id
        # Strategy 1: convert_tokens_to_ids (works if it's a single vocab entry)
        tid = self.tokenizer.convert_tokens_to_ids(primary)
        if tid != unk:
            print(f"  [token] '{primary}' -> {tid} (via convert_tokens_to_ids)")
            return tid
        # Strategy 2: encode and take last token (handles BPE splits)
        encoded = self.tokenizer.encode(primary, add_special_tokens=False)
        if encoded:
            tid = encoded[-1]  # last token usually captures the word
            print(f"  [token] '{primary}' -> {tid} (via encode, full={encoded})")
            return tid
        # Strategy 3: try fallbacks
        for fb in fallbacks:
            tid = self.tokenizer.convert_tokens_to_ids(fb)
            if tid != unk:
                print(f"  [token] '{primary}' failed, using '{fb}' -> {tid}")
                return tid
            encoded = self.tokenizer.encode(fb, add_special_tokens=False)
            if encoded:
                tid = encoded[-1]
                print(f"  [token] '{primary}' failed, using '{fb}' -> {tid} (via encode)")
                return tid
        raise ValueError(f"Could not resolve any token ID for '{primary}' or fallbacks {fallbacks}")

    @torch.no_grad()
    def predict(self, pairs: list[tuple[str, str]], batch_size: int = 8,
                instruction: str = "Given a query, determine if the document is relevant.") -> list[float]:
        """Score (query, doc) pairs. Returns P(yes) for each pair."""
        all_scores = []
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i + batch_size]
            messages_batch = []
            for query, doc in batch:
                msg = [
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": f"Query: {query}\nDocument: {doc[:1500]}"}
                ]
                messages_batch.append(
                    self.tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
                )
            inputs = self.tokenizer(
                messages_batch, return_tensors="pt", padding=True, truncation=True, max_length=2048
            ).to(self.device)
            outputs = self.model(**inputs)
            logits = outputs.logits[:, -1, :]
            yes_no_logits = logits[:, [self.no_id, self.yes_id]]
            probs = F.softmax(yes_no_logits, dim=-1)
            scores = probs[:, 1].cpu().tolist()
            all_scores.extend(scores)
        return all_scores


# RERANKER DISABLED - Qwen3-Reranker produces uniform scores (~0.01),
# nothing passes any meaningful cutoff. Using RRF scores directly in aggregate_and_output.
# Skipping model load saves ~1.2GB VRAM on GPU 1.
reranker = None
print("Reranker: DISABLED (using RRF scores directly - saves 1.2GB VRAM)")
print(f"  GPU 1 memory: {torch.cuda.memory_allocated(1)/1e9:.2f} GB allocated")


Reranker: DISABLED (using RRF scores directly - saves 1.2GB VRAM)
  GPU 1 memory: 1.20 GB allocated


In [31]:
# Cell 9: Build BM25 Indices
# MEMORY OPTIMIZATION: 
#   1. Delete tokenized corpus lists after BM25 internalizes them (~500-1000MB freed)
#   2. Truncate raw document text to snippet length (200 chars) â€” full text already 
#      captured in citation_to_text (1500 chars) for reranker use.
#      filtered_hybrid_search() only needs doc["text"][:200] for snippets.
from rank_bm25 import BM25Okapi

_re_token = re.compile(r"[a-z\u00e4\u00f6\u00fc\u00df\d]+")


def tokenize_german(text: str) -> list[str]:
    """Simple German tokenizer for BM25."""
    tokens = _re_token.findall(text.lower())
    return [t for t in tokens if len(t) > 2]


print("Building BM25 indices...")
t0 = time.time()

# Laws BM25
_law_bm25_corpus = [tokenize_german(f"{d['citation']} {d['text']}") for d in laws_documents]
bm25_law_index = BM25Okapi(_law_bm25_corpus)
del _law_bm25_corpus  # BM25 already internalized â€” free the token lists
print(f"  Laws BM25: {bm25_law_index.corpus_size:,} docs ({time.time()-t0:.1f}s)")

t1 = time.time()
_court_bm25_corpus = [tokenize_german(f"{d['citation']} {d['text']}") for d in courts_documents]
bm25_court_index = BM25Okapi(_court_bm25_corpus)
del _court_bm25_corpus  # BM25 already internalized â€” free the token lists
print(f"  Courts BM25: {bm25_court_index.corpus_size:,} docs ({time.time()-t1:.1f}s)")

# â”€â”€â”€ Truncate raw document text to snippet length â”€â”€â”€
# After this point, only filtered_hybrid_search() uses these lists (for 200-char snippets).
# Full 1500-char text for reranker is already in citation_to_text (built in Cell 5).
_truncated = 0
for doc in laws_documents:
    if len(doc["text"]) > 200:
        doc["text"] = doc["text"][:200]
        _truncated += 1
for doc in courts_documents:
    if len(doc["text"]) > 200:
        doc["text"] = doc["text"][:200]
        _truncated += 1

gc.collect()
print(f"  âœ“ BM25 token lists freed + {_truncated:,} docs truncated to 200 chars")
print(f"  Total BM25 build: {time.time()-t0:.1f}s")

Building BM25 indices...
  Laws BM25: 175,933 docs (4.4s)
  Courts BM25: 200,000 docs (5.7s)
  âœ“ BM25 token lists freed + 0 docs truncated to 200 chars
  Total BM25 build: 10.5s


In [32]:
# Cell 9.5: Load LLM (Mistral-7B on GPU 0)
# DEFERRED to after all index building (FAISS + BM25) to free ~2GB RAM during encoding.
# LLM is first used in Cell 13 (Planner). No point loading it during FAISS/BM25 build.
from llama_cpp import Llama

model_file = MODEL_PATH / CONFIG["model_file"]
if not model_file.exists():
    gguf_files = list(MODEL_PATH.rglob("*.gguf"))
    if gguf_files:
        model_file = gguf_files[0]
    else:
        raise FileNotFoundError(f"No GGUF model found in {MODEL_PATH}")

print(f"Loading LLM: {model_file.name}")
llm = Llama(
    model_path=str(model_file),
    n_ctx=CONFIG["n_ctx"],
    n_threads=CONFIG["n_threads"],
    n_gpu_layers=CONFIG["n_gpu_layers"],
    main_gpu=0,
    verbose=False,
)
print(f"  LLM loaded: n_ctx={CONFIG['n_ctx']}, GPU 0")
print(f"  âœ“ All indices built BEFORE LLM â€” no memory contention during embedding")

Loading LLM: mistral-7b-instruct-v0.2.Q4_K_M.gguf


llama_context: n_ctx_seq (16384) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


  LLM loaded: n_ctx=16384, GPU 0
  âœ“ All indices built BEFORE LLM â€” no memory contention during embedding


In [44]:
# Cell 10: Filtered Hybrid Search

def embed_query(query: str, corpus: str = "laws") -> np.ndarray:
    """Embed a search query with instruction prefix."""
    prompt = CONFIG["prompt_query_law"] if corpus == "laws" else CONFIG["prompt_query_court"]
    vec = _st_model.encode([query], prompt=prompt, normalize_embeddings=True)
    return vec.astype("float32")


def reciprocal_rank_fusion(rankings: list[list[tuple[int, float]]], k: int = 60) -> list[tuple[int, float]]:
    """Fuse multiple ranked lists using RRF."""
    scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, (doc_id, _score) in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def filtered_hybrid_search(
    query: str,
    corpus: str,
    filter_codes: list[str],
    top_k: int = 10,
) -> list[tuple[str, float, str]]:
    """
    Hybrid search with metadata filtering.
    Returns: [(citation, score, snippet), ...]
    """
    if corpus == "both":
        law_codes = [c for c in filter_codes if c in law_code_to_indices]
        court_codes = [c for c in filter_codes if c in court_code_to_indices]
        # If no valid codes for a corpus, search it unfiltered (not skip it)
        r_law = filtered_hybrid_search(query, "laws", law_codes, top_k)
        r_court = filtered_hybrid_search(query, "courts", court_codes, top_k)
        # Interleave and cap
        combined = sorted(r_law + r_court, key=lambda x: x[1], reverse=True)
        return combined[:top_k]

    # Select corpus-specific resources
    if corpus == "laws":
        documents = laws_documents
        code_map = law_code_to_indices
        faiss_index = faiss_law_index
        bm25_index = bm25_law_index
    else:
        documents = courts_documents
        code_map = court_code_to_indices
        faiss_index = faiss_court_index
        bm25_index = bm25_court_index

    # Build valid index set from filter codes
    valid_indices = None
    if filter_codes:
        arrays = [code_map[c] for c in filter_codes if c in code_map]
        if arrays:
            valid_indices = np.unique(np.concatenate(arrays))

    # --- FAISS search (with filtering) ---
    query_vec = embed_query(query, corpus)
    faiss_top_k = top_k * 3  # Over-fetch for fusion

    if valid_indices is not None and len(valid_indices) > 0:
        # Use IDSelectorArray for filtered search
        id_selector = faiss.IDSelectorArray(len(valid_indices), faiss.swig_ptr(valid_indices))
        params = faiss.SearchParameters(sel=id_selector)
        scores_f, ids_f = faiss_index.search(query_vec, faiss_top_k, params=params)
    else:
        scores_f, ids_f = faiss_index.search(query_vec, faiss_top_k)

    faiss_results = [(int(ids_f[0][i]), float(scores_f[0][i]))
                     for i in range(len(ids_f[0])) if ids_f[0][i] >= 0]

    # --- BM25 search (with post-filtering) ---
    query_tokens = tokenize_german(query)
    bm25_scores = bm25_index.get_scores(query_tokens)
    bm25_ranked = np.argsort(bm25_scores)[::-1][:top_k * 5]

    if valid_indices is not None:
        valid_set = set(valid_indices.tolist())
        bm25_results = [(int(idx), float(bm25_scores[idx]))
                        for idx in bm25_ranked if int(idx) in valid_set][:top_k * 3]
    else:
        bm25_results = [(int(idx), float(bm25_scores[idx])) for idx in bm25_ranked[:top_k * 3]]

    # --- RRF Fusion ---
    combined = reciprocal_rank_fusion([faiss_results, bm25_results], k=CONFIG["rrf_k"])

    # --- ADAPTIVE FALLBACK: if <5 results with filter, broaden ---
    if len(combined) < CONFIG["adaptive_fallback_threshold"] and valid_indices is not None:
        return filtered_hybrid_search(query, corpus, [], top_k)

    # Format output
    results = []
    for doc_id, score in combined[:top_k]:
        doc = documents[doc_id]
        results.append((doc["citation"], score, doc["text"][:200]))
    return results


# Quick test
test_results = filtered_hybrid_search("Untersuchungshaft HaftgrÃ¼nde", "laws", ["StPO"], top_k=5)
print("Test search (StPO, 'Untersuchungshaft HaftgrÃ¼nde'):")
for cit, score, snip in test_results:
    print(f"  {cit} ({score:.4f})")

Test search (StPO, 'Untersuchungshaft HaftgrÃ¼nde'):
  Art. 226 Abs. 3 StPO (0.0305)
  Art. 226 Abs. 4 StPO (0.0303)
  Art. 234 Abs. 1 StPO (0.0164)
  Art. 221 Abs. 1 StPO (0.0161)
  Art. 229 Abs. 3 StPO (0.0161)


In [45]:
# Cell 11: Load Context Files + Prompts + Grammars

# All context/prompt files are uploaded flat to rag-checkpoints (no subfolders)
# Files: planner_system.txt, executor_system.txt, executor_procedural.txt,
#        planner.gbnf, executor.gbnf, fallback_rules.txt,
#        swiss_legal_system.txt, routing_guide_laws.txt, routing_guide_courts.txt,
#        terminology_bridge.txt, procedural_defaults.txt

def load_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def _find_file(filename: str) -> Path:
    """Find a file in CHECKPOINT_DS (flat), or local repo structure."""
    # 1. Flat in checkpoint dataset (Kaggle upload â€” no subfolders)
    flat = CHECKPOINT_DS / filename
    if flat.exists():
        return flat
    # 2. Local repo structure (for local dev)
    NOTEBOOK_DIR = Path(".").resolve()
    for base in [NOTEBOOK_DIR / "..", NOTEBOOK_DIR, Path("/kaggle/working/repo")]:
        for subdir in ["prompts", "context", ""]:
            p = base / subdir / filename if subdir else base / filename
            if p.exists():
                return p
    raise FileNotFoundError(f"Cannot find '{filename}' in CHECKPOINT_DS or local paths")


# Load all context files
swiss_legal_system = load_text(_find_file("swiss_legal_system.txt"))
routing_guide_laws = load_text(_find_file("routing_guide_laws.txt"))
routing_guide_courts = load_text(_find_file("routing_guide_courts.txt"))
terminology_bridge = load_text(_find_file("terminology_bridge.txt"))
procedural_defaults_text = load_text(_find_file("procedural_defaults.txt"))

# Load prompts
planner_system_template = load_text(_find_file("planner_system.txt"))
executor_system_template = load_text(_find_file("executor_system.txt"))
executor_procedural_template = load_text(_find_file("executor_procedural.txt"))

# Load GBNF grammars
from llama_cpp import LlamaGrammar
planner_grammar = LlamaGrammar.from_string(load_text(_find_file("planner.gbnf")))
executor_grammar = LlamaGrammar.from_string(load_text(_find_file("executor.gbnf")))

print(f"Context + prompts loaded (flat from {CHECKPOINT_DS})")
print(f"  swiss_legal_system: {len(swiss_legal_system):,} chars")
print(f"  routing_guide_laws: {len(routing_guide_laws):,} chars")
print(f"  routing_guide_courts: {len(routing_guide_courts):,} chars")
print(f"  terminology_bridge: {len(terminology_bridge):,} chars")
print(f"Prompts + grammars loaded.")

Context + prompts loaded (flat from /kaggle/input/datasets/charan1996/rag-checkpoints-v2)
  swiss_legal_system: 7,215 chars
  routing_guide_laws: 22,586 chars
  routing_guide_courts: 10,890 chars
  terminology_bridge: 8,062 chars
Prompts + grammars loaded.


In [46]:
# Cell 12: Taxonomy Cache for Executor
# Parses routing guides into per-code sections for targeted injection

_TAXONOMY_CACHE: dict[str, str] = {}
_TAXONOMY_CODE_RE = re.compile(r"[\u2022\u00b7]\s*(\S+)\s*\(")


def _build_taxonomy_cache() -> dict[str, str]:
    """Parse routing guides into per-code taxonomy sections."""
    cache: dict[str, str] = {}
    for guide_text in [routing_guide_laws, routing_guide_courts]:
        lines = guide_text.split("\n")
        current_code = None
        current_lines: list[str] = []
        for line in lines:
            m = _TAXONOMY_CODE_RE.match(line.strip())
            if m:
                # Save previous
                if current_code and current_lines:
                    cache[current_code] = "\n".join(current_lines)
                current_code = m.group(1)
                current_lines = [line.strip()]
            elif current_code and line.strip():
                current_lines.append(line.strip())
            elif current_code and not line.strip():
                # Empty line = end of section
                if current_lines:
                    cache[current_code] = "\n".join(current_lines)
                current_code = None
                current_lines = []
        # Final section
        if current_code and current_lines:
            cache[current_code] = "\n".join(current_lines)
    return cache


_TAXONOMY_CACHE = _build_taxonomy_cache()


def get_taxonomy_section(filter_codes: list[str]) -> str:
    """Get combined taxonomy text for given filter codes."""
    sections = [_TAXONOMY_CACHE[c] for c in filter_codes if c in _TAXONOMY_CACHE]
    if not sections:
        return ""
    return "TAXONOMIE DIESER RICHTUNG:\n" + "\n\n".join(sections)


print(f"Taxonomy cache: {len(_TAXONOMY_CACHE)} codes")
print(f"Sample: {list(_TAXONOMY_CACHE.keys())[:8]}")

Taxonomy cache: 64 codes
Sample: ['StGB', 'JStG', 'MStG', 'StPO', 'JStPO', 'StBOG', 'MStP', 'ZGB']


In [ ]:
# Cell 13: Planner Agent + Dynamic Context Selection
# Keyword matching selects relevant SECTIONS from routing_guide_laws + terminology_bridge
# so we stay well within 16K context window.
# Coverage: ALL 48 law codes mapped via exhaustive English+German keywords.

from typing import Optional
from dataclasses import dataclass, field
import re as _re


@dataclass
class Direction:
    priority: int
    corpus: str  # "laws" | "courts" | "both"
    rechtsgebiet: str
    filter_codes: list[str]
    reasoning: str
    seed_queries: list[str]


@dataclass
class Plan:
    sachverhalt: str
    rechtsfragen: list[str]
    directions: list[Direction]


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# SECTION PARSING: Split routing guides and terminology into sections
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def _parse_sections_by_marker(text: str, marker: str = "===") -> dict[str, str]:
    """Parse text into sections by === HEADER === or --- HEADER --- markers.
    Returns dict: section_key â†’ section_text (including the header line).
    """
    sections = {}
    lines = text.split("\n")
    if marker == "===":
        header_re = _re.compile(r"^===\s*(.+?)\s*===\s*$")
    else:
        header_re = _re.compile(r"^---\s*(.+?)\s*---\s*$")
    
    preamble_key = "__PREAMBLE__" if marker == "===" else "__INTRO__"
    current_key = preamble_key
    current_lines = []
    
    for line in lines:
        m = header_re.match(line)
        if m:
            if current_lines:
                sections[current_key] = "\n".join(current_lines).strip()
            current_key = m.group(1).strip()
            current_lines = [line]
        else:
            current_lines.append(line)
    
    if current_lines:
        sections[current_key] = "\n".join(current_lines).strip()
    
    return sections


def _find_section_keys(pool: dict[str, str], substrings: list[str]) -> list[str]:
    """Find keys in a parsed section pool whose key contains any substring (case-insensitive)."""
    matches = []
    for key in pool:
        key_lower = key.lower()
        if any(sub.lower() in key_lower for sub in substrings):
            matches.append(key)
    return matches


# â”€â”€â”€ Parse all three context files into sections â”€â”€â”€
_ROUTING_SECTIONS = _parse_sections_by_marker(routing_guide_laws, "===")
_COURT_SECTIONS = _parse_sections_by_marker(routing_guide_courts, "===")
_TERMINOLOGY_SECTIONS = _parse_sections_by_marker(terminology_bridge, "---")

# â”€â”€â”€ Laws: Map domains â†’ routing + terminology section keys â”€â”€â”€
_DOMAIN_ROUTING_MATCHERS: dict[str, list[str]] = {
    "STRAFRECHT": ["strafrecht"],
    "STRAFPROZESS": ["strafprozess", "strafrecht"],
    "ZIVILRECHT": ["zivilrecht"],
    "PROZESSRECHT": ["prozessrecht"],
    "SOZIALVERSICHERUNG": ["sozialversicherung"],
    "OEFFENTLICHES_RECHT": ["Ã¶ffentlich"],
    "STEUERRECHT": ["steuer", "weitere"],
    "FINANZMARKTRECHT": ["finanzmarkt", "weitere"],
    "WEITERE": ["weitere"],
}

_DOMAIN_TERMINOLOGY_MATCHERS: dict[str, list[str]] = {
    "STRAFRECHT": ["strafrecht", "criminal law"],
    "STRAFPROZESS": ["strafprozess", "criminal procedure"],
    "ZIVILRECHT": ["zivilrecht", "civil law", "familienrecht", "family", "erbrecht", "succession", "sachenrecht", "property"],
    "PROZESSRECHT": ["verfahrensrecht", "procedural"],
    "SOZIALVERSICHERUNG": ["sozialversicherung", "social insurance"],
    "OEFFENTLICHES_RECHT": ["Ã¶ffentlich", "public law"],
    "STEUERRECHT": [],
    "FINANZMARKTRECHT": [],
    "WEITERE": [],
}

_DOMAIN_TO_SECTIONS: dict[str, dict] = {}
for _dom, _matchers in _DOMAIN_ROUTING_MATCHERS.items():
    _rkeys = _find_section_keys(_ROUTING_SECTIONS, _matchers)
    _tmatchers = _DOMAIN_TERMINOLOGY_MATCHERS.get(_dom, [])
    _tkeys = _find_section_keys(_TERMINOLOGY_SECTIONS, _tmatchers)
    _DOMAIN_TO_SECTIONS[_dom] = {"routing_keys": _rkeys, "terminology_keys": _tkeys}

# Laws header (classification rules â€” always useful, short)
_HEADER_KEYS = _find_section_keys(_ROUTING_SECTIONS, ["routing", "gesetzes-routing", "klassifikation"])
if not _HEADER_KEYS:
    _HEADER_KEYS = [list(_ROUTING_SECTIONS.keys())[0]] if _ROUTING_SECTIONS else []

# â”€â”€â”€ Courts: Map court divisions â†’ court section keys (EXACT keys, no substring matching) â”€â”€â”€
# Using exact section header strings from routing_guide_courts.txt to avoid
# substring ambiguity (e.g. "i. Ã¶ffentlich" matching "ii. Ã¶ffentlich...")
_COURT_TO_SECTIONS: dict[str, list[str]] = {
    "COURT_STRAFPROZESS": ["I. Ã–FFENTLICH-RECHTLICHE ABTEILUNG"],       # 1B_ detention/coercive
    "COURT_VERWALTUNG":   ["I. Ã–FFENTLICH-RECHTLICHE ABTEILUNG"],       # 1C_ admin/planning
    "COURT_OEFFENTLICH":  ["II. Ã–FFENTLICH-RECHTLICHE ABTEILUNG"],      # 2C_ foreigners/tax/health
    "COURT_VERTRAG":      ["ZIVILRECHTLICHE ABTEILUNGEN"],                  # 4A_ contracts
    "COURT_FAMILIE":      ["ZIVILRECHTLICHE ABTEILUNGEN"],                  # 5A_ family/inheritance
    "COURT_ZIVIL":        ["ZIVILRECHTLICHE ABTEILUNGEN"],                  # 4A_+5A_ combined
    "COURT_STRAF":        ["STRAFRECHTLICHE ABTEILUNG"],                    # 6B_ criminal
    "COURT_SOZIAL":       ["SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN"],    # 8C_+9C_ combined
    "COURT_SOZIAL_IV":    ["SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN"],    # 8C_ IV/UV/ALV
    "COURT_SOZIAL_RENTEN":["SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN"],    # 9C_ AHV/KV/EL
    "COURT_BGE_STRAF":       ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],  # BGE_IV
    "COURT_BGE_ZIVIL":       ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],  # BGE_III
    "COURT_BGE_SOZIAL":      ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],  # BGE_V
    "COURT_BGE_OEFFENTLICH": ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],  # BGE_I/II
    "COURT_BUNDESSTRAF":     ["BUNDESSTRAFGERICHT (Beschwerdekammer & Berufungskammer)"],  # 7B_
}

# Court header (classification rules â€” always useful)
_COURT_HEADER_KEYS = ["GERICHTS-ROUTING (search_courts)"]

# â”€â”€â”€ Map law domains â†’ court divisions (for cross-linking) â”€â”€â”€
_DOMAIN_TO_COURT_DIVISIONS: dict[str, list[str]] = {
    "STRAFRECHT": ["COURT_STRAF", "COURT_BGE_STRAF"],
    "STRAFPROZESS": ["COURT_STRAFPROZESS", "COURT_STRAF", "COURT_BGE_STRAF", "COURT_BUNDESSTRAF"],
    "ZIVILRECHT": ["COURT_VERTRAG", "COURT_FAMILIE", "COURT_BGE_ZIVIL"],
    "PROZESSRECHT": ["COURT_VERTRAG", "COURT_STRAF", "COURT_BGE_OEFFENTLICH"],
    "SOZIALVERSICHERUNG": ["COURT_SOZIAL_IV", "COURT_SOZIAL_RENTEN", "COURT_BGE_SOZIAL"],
    "OEFFENTLICHES_RECHT": ["COURT_OEFFENTLICH", "COURT_VERWALTUNG", "COURT_BGE_OEFFENTLICH"],
    "STEUERRECHT": ["COURT_OEFFENTLICH", "COURT_BGE_OEFFENTLICH"],
    "FINANZMARKTRECHT": ["COURT_OEFFENTLICH", "COURT_BGE_OEFFENTLICH"],
    "WEITERE": ["COURT_OEFFENTLICH", "COURT_STRAF", "COURT_BUNDESSTRAF"],
}

print(f"Laws routing sections: {list(_ROUTING_SECTIONS.keys())}")
print(f"Court routing sections: {list(_COURT_SECTIONS.keys())}")
print(f"  Court sizes: { {k: len(v) for k, v in _COURT_SECTIONS.items()} }")
print(f"Terminology sections: {list(_TERMINOLOGY_SECTIONS.keys())}")
print(f"Laws header: {_HEADER_KEYS}")
print(f"Court header: {_COURT_HEADER_KEYS}")
print(f"Domainâ†’Laws mapping:")
for _d, _s in _DOMAIN_TO_SECTIONS.items():
    print(f"  {_d}: routing={_s['routing_keys']}, terminology={[k[:25] for k in _s['terminology_keys']]}")
print(f"Domainâ†’Courts mapping:")
for _d, _cdivs in _DOMAIN_TO_COURT_DIVISIONS.items():
    _all_ckeys = []
    for _cd in _cdivs:
        _all_ckeys.extend(_COURT_TO_SECTIONS.get(_cd, []))
    print(f"  {_d}: {_all_ckeys}")


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# EXHAUSTIVE KEYWORD MAPPING: Every domain covered by English + German keywords
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_DOMAIN_KEYWORDS: dict[str, list[str]] = {
    # â•â•â• STRAFRECHT (StGB, JStG, MStG) â•â•â•
    "STRAFRECHT": [
        # English â€” crimes & criminal law concepts
        "murder", "homicide", "manslaughter", "killing", "assault", "battery",
        "bodily harm", "injury", "theft", "robbery", "burglary", "fraud",
        "embezzlement", "forgery", "arson", "kidnapping", "rape", "sexual",
        "drug", "narcotic", "money laundering", "criminal", "crime", "offense",
        "offence", "felony", "misdemeanor", "guilty", "culpable", "intentional",
        "negligent", "reckless", "attempt", "complicity",
        "accomplice", "aiding", "abetting", "incitement", "conspiracy",
        "self-defense", "self-defence", "sentence", "sentencing", "penalty",
        "punishment", "fine", "imprisonment", "prison", "probation", "parole",
        "recidivism", "juvenile", "minor", "youth",
        "military criminal", "desertion",
        "threat", "extortion", "bribery", "corruption", "trafficking",
        "stalking", "harassment", "vandalism", "damage to property",
        "domestic violence", "weapon", "firearm",
        # German
        "strafrecht", "strafbar", "strafe", "straftat", "delikt", "vergehen",
        "verbrechen", "tÃ¶tung", "mord", "totschlag", "kÃ¶rperverletzung",
        "diebstahl", "raub", "betrug", "arglist", "veruntreuung",
        "urkundenfÃ¤lschung", "brandstiftung", "entfÃ¼hrung", "vergewaltigung",
        "drohung", "nÃ¶tigung", "erpressung", "bestechung",
        "freiheitsstrafe", "geldstrafe", "bedingt", "bewÃ¤hrung",
        "strafzumessung", "landesverweisung", "vorsatz", "fahrlÃ¤ssigkeit",
        "versuch", "teilnahme", "gehilfenschaft", "anstiftung",
        "notwehr", "notstand", "schuldfÃ¤higkeit",
        "jugendstrafrecht", "minderjÃ¤hrig",
        "militÃ¤rstrafrecht",
        "stgb", "jstg", "mstg",
    ],
    
    # â•â•â• STRAFPROZESS (StPO, JStPO, StBOG, MStP) â•â•â•
    "STRAFPROZESS": [
        # English
        "detention", "pre-trial", "pretrial", "remand", "custody",
        "arrest", "warrant", "search", "seizure", "wiretap", "surveillance",
        "prosecution", "prosecutor", "indictment", "investigation",
        "evidence", "admissibility", "confession", "witness", "testimony",
        "bail", "release", "flight risk", "collusion", "risk of reoffending",
        "criminal procedure", "criminal process",
        "right to silence", "right to counsel", "defense lawyer",
        "plea", "acquittal", "conviction",
        # German
        "strafprozess", "strafverfahren", "untersuchungshaft",
        "sicherheitshaft", "haftgrund", "fluchtgefahr", "kollusionsgefahr",
        "wiederholungsgefahr", "verhÃ¤ltnismÃ¤ssigkeit",
        "Ã¼berhaft", "haftprÃ¼fung", "haftverlÃ¤ngerung", "haftentlassung",
        "staatsanwaltschaft", "anklage",
        "beweisverwertung", "beweisverwertungsverbot",
        "einvernahme", "hausdurchsuchung", "beschlagnahme",
        "zwangsmassnahme", "ersatzmassnahme",
        "jugendstrafverfahren", "jugendanwalt",
        "bundesstrafgericht", "bundesanwaltschaft",
        "stpo", "jstpo", "stbog", "mstp",
        "haft", "inhaftierung",
    ],
    
    # â•â•â• ZIVILRECHT (ZGB, OR, GBV, IPRG, SchKG, URG, DSG) â•â•â•
    "ZIVILRECHT": [
        # English â€” contracts, family, property, succession, debt, IP, data
        "contract", "agreement", "obligation", "breach", "performance",
        "non-performance", "damages", "compensation", "liability",
        "tort", "fault", "causation", "warranty", "guarantee",
        "sale", "purchase", "lease", "rent", "tenancy",
        "employment", "worker", "employee", "employer", "dismissal",
        "termination", "notice period", "wrongful termination",
        "marriage", "divorce", "separation", "alimony", "maintenance",
        "child support", "spousal support", "custody", "parental",
        "visitation", "best interest", "adoption", "prenuptial",
        "matrimonial", "property regime",
        "inheritance", "succession", "will", "testament", "heir",
        "estate", "probate", "bequest", "forced heirship", "disinheritance",
        "property", "ownership", "possession", "real estate", "land",
        "mortgage", "easement", "servitude", "land register", "condominium",
        "debt", "creditor", "debtor", "bankruptcy", "insolvency",
        "enforcement", "garnishment", "attachment", "debt collection", "foreclosure",
        "copyright", "intellectual property", "author",
        "data protection", "privacy", "personal data",
        "international private law", "conflict of laws", "applicable law",
        "choice of law", "foreign judgment", "recognition",
        # German
        "zivilrecht", "privatrecht", "vertrag", "obligation",
        "kaufvertrag", "werkvertrag", "auftrag", "miete", "mietvertrag",
        "arbeitsvertrag", "kÃ¼ndigung", "arbeitsrecht", "arbeitnehmer",
        "schadenersatz", "gewÃ¤hrleistung", "verzug", "erfÃ¼llung",
        "nichterfÃ¼llung", "mangel", "haftpflicht", "haftung", "verschulden",
        "ehe", "scheidung", "unterhalt", "kindesunterhalt",
        "sorgerecht", "elterliche sorge", "obhut", "kindeswohl",
        "erbrecht", "erbschaft", "testament", "pflichtteil", "enterbung",
        "nachlass", "erbfolge",
        "eigentum", "besitz", "grundbuch", "grundpfand", "hypothek",
        "dienstbarkeit", "stockwerkeigentum",
        "betreibung", "pfÃ¤ndung", "konkurs", "rechtsÃ¶ffnung",
        "verlustschein", "existenzminimum", "zahlungsbefehl",
        "schuld", "glÃ¤ubiger", "schuldner",
        "urheberrecht", "datenschutz", "personendaten",
        "internationales privatrecht", "anwendbares recht",
        "zgb", "or", "gbv", "iprg", "schkg", "urg", "dsg",
    ],
    
    # â•â•â• PROZESSRECHT (ZPO, BGG, VwVG) â•â•â•
    "PROZESSRECHT": [
        # English
        "procedure", "procedural", "civil procedure",
        "federal court", "supreme court", "federal tribunal",
        "appeal", "cassation", "complaint", "legal remedy",
        "jurisdiction", "competence", "standing", "locus standi",
        "time limit", "deadline", "statute of limitations", "limitation",
        "burden of proof", "standard of proof",
        "injunction", "provisional measure", "interim measure",
        "res judicata", "arbitration", "mediation", "conciliation",
        "costs", "court costs", "legal aid",
        "administrative procedure", "administrative law",
        "right to be heard", "due process", "fair trial",
        # German
        "prozessrecht", "verfahrensrecht", "zivilprozess",
        "klage", "berufung", "beschwerde", "rechtsmittel",
        "zustÃ¤ndigkeit", "legitimation", "streitwert",
        "frist", "beschwerdefrist", "verwirkung",
        "beweislast", "beweismass",
        "vorsorgliche massnahme", "superprovisorisch",
        "schlichtung", "schiedsverfahren",
        "bundesgericht", "bundesgerichtsgesetz",
        "verwaltungsverfahren", "verfÃ¼gung", "einsprache",
        "akteneinsicht", "begrÃ¼ndungspflicht", "rechtliches gehÃ¶r",
        "willkÃ¼r", "willkÃ¼rverbot",
        "zpo", "bgg", "vwvg",
    ],
    
    # â•â•â• SOZIALVERSICHERUNG (ATSG, IVG, UVG, KVG, BVG, AVIG, AHVG) â•â•â•
    "SOZIALVERSICHERUNG": [
        # English
        "disability", "disabled", "invalid", "invalidity", "incapacity",
        "pension", "retirement", "old age", "social security",
        "social insurance", "welfare",
        "accident", "occupational disease", "work accident",
        "unemployment", "unemployed", "job seeker",
        "health insurance", "medical", "hospital", "treatment",
        "rehabilitation", "reintegration", "vocational training",
        "work capacity", "earning capacity", "residual capacity",
        "medical assessment", "expert opinion",
        "benefit", "allowance", "daily allowance",
        "contribution", "premium", "deductible",
        "survivors", "widow", "orphan",
        # German
        "sozialversicherung", "sozialversicherungsrecht",
        "invaliditÃ¤t", "invalidenrente", "invaliditÃ¤tsgrad",
        "eingliederung", "eingliederungsmassnahme", "iv-stelle",
        "restarbeitsfÃ¤higkeit", "arbeitsfÃ¤higkeit", "arbeitsunfÃ¤higkeit",
        "einkommensvergleich", "tabellenlohn", "begutachtung",
        "unfall", "unfallversicherung", "berufskrankheit",
        "kausalitÃ¤t", "adÃ¤quanz", "integritÃ¤tsentschÃ¤digung", "taggeld",
        "krankenversicherung", "krankenpflege", "prÃ¤mie", "franchise",
        "pensionskasse", "berufliche vorsorge", "freizÃ¼gigkeit",
        "altersrente", "Ã¼berentschÃ¤digung", "austrittsleistung",
        "arbeitslosenentschÃ¤digung", "kurzarbeit", "einstelltage",
        "vermittlungsfÃ¤higkeit", "rahmenfrist",
        "ahv", "hinterlassenenrente", "beitragspflicht",
        "ergÃ¤nzungsleistung", "hilflosenentschÃ¤digung",
        "atsg", "ivg", "uvg", "kvg", "bvg", "avig", "ahvg",
    ],
    
    # â•â•â• Ã–FFENTLICHES RECHT (BV, AIG, AsylG, RPG, USG, NHG) â•â•â•
    "OEFFENTLICHES_RECHT": [
        # English
        "constitutional", "constitution", "fundamental right", "human right",
        "equal treatment", "equality", "discrimination",
        "freedom", "liberty", "personal freedom", "expression",
        "proportionality", "arbitrary", "arbitrariness",
        "foreigner", "alien", "immigrant", "immigration", "migration",
        "residence permit", "settlement", "family reunification",
        "deportation", "expulsion", "removal",
        "hardship case", "integration",
        "asylum", "refugee", "persecution", "protection",
        "temporary admission", "dublin",
        "planning", "zoning", "building permit", "construction",
        "land use", "spatial planning",
        "environment", "environmental", "pollution", "emission",
        "noise", "waste", "contaminated site",
        "nature", "landscape", "conservation", "heritage",
        # German
        "Ã¶ffentliches recht", "verfassungsrecht", "grundrecht",
        "gleichbehandlung", "rechtsgleichheit",
        "willkÃ¼rverbot", "persÃ¶nliche freiheit", "meinungsfreiheit",
        "eigentumsgarantie", "bundesverfassung",
        "auslÃ¤nder", "aufenthaltsbewilligung", "niederlassungsbewilligung",
        "wegweisung", "familiennachzug", "hÃ¤rtefall",
        "asyl", "flÃ¼chtling", "verfolgung", "vorlÃ¤ufige aufnahme",
        "unzumutbarkeit", "asylverfahren",
        "raumplanung", "nutzungszone", "baubewilligung",
        "zonenkonformitÃ¤t", "ausnahmebewilligung",
        "umweltschutz", "umweltvertrÃ¤glichkeit", "immissionen",
        "emissionen", "lÃ¤rm", "altlasten",
        "naturschutz", "landschaftsschutz", "heimatschutz",
        "bv", "aig", "asylg", "rpg", "usg", "nhg",
    ],
    
    # â•â•â• STEUERRECHT (DBG, VStG, MWSTG) â•â•â•
    "STEUERRECHT": [
        # English
        "tax", "taxation", "taxpayer", "income tax", "corporate tax",
        "profit tax", "capital gains", "withholding tax",
        "dividend", "interest income", "deduction", "tax deduction",
        "taxable income", "assessment", "tax return",
        "tax evasion", "tax fraud", "tax avoidance", "double taxation",
        "vat", "value added tax", "sales tax",
        "transfer pricing", "international tax",
        # German
        "steuerrecht", "steuer", "steuerpflicht",
        "einkommenssteuer", "gewinnsteuer", "kapitalgewinn",
        "abzug", "steuerbares einkommen", "veranlagung",
        "steuerhinterziehung", "steuerbetrug",
        "verrechnungssteuer", "rÃ¼ckerstattung", "meldeverfahren",
        "dividende", "kapitalertrag",
        "mehrwertsteuer", "vorsteuer", "umsatz",
        "doppelbesteuerung",
        "dbg", "vstg", "mwstg",
    ],
    
    # â•â•â• FINANZMARKTRECHT (FIDLEG, FINMAG, FINIG, BankG, BEG, FINMA) â•â•â•
    "FINANZMARKTRECHT": [
        # English
        "financial", "finance", "bank", "banking", "credit",
        "investment", "investor", "fund", "asset management",
        "securities", "stock", "bond", "share", "equity",
        "market abuse", "insider trading", "insider",
        "supervision", "regulatory", "license",
        "capital requirement", "liquidity", "solvency",
        "prospectus", "disclosure", "transparency",
        "money laundering", "aml", "compliance", "due diligence",
        "fintech", "crypto", "blockchain",
        "deposit insurance", "bank secrecy",
        "financial service", "advisory", "portfolio",
        # German
        "finanzmarkt", "finanzmarktrecht", "finanzdienstleistung",
        "banken", "bankbewilligung", "bankgeheimnis",
        "eigenmittel", "liquiditÃ¤t", "einlagensicherung",
        "finma", "bewilligungspflicht", "aufsicht",
        "marktmissbrauch", "insiderhandel",
        "anlageberatung", "prospekt", "prospektpflicht",
        "vermÃ¶gensverwalter", "fondsleitung", "wertpapierhaus",
        "bucheffekte", "effektenkonto",
        "geldwÃ¤scherei", "sorgfaltspflicht",
        "fidleg", "finmag", "finig", "bankg", "beg",
    ],
    
    # â•â•â• WEITERE (SVG, IRSG, UWG, KGTG, ParlG, LwG, HMG, LFG, EBG) â•â•â•
    "WEITERE": [
        # English
        "traffic", "road", "driving", "driver", "license revocation",
        "speeding", "drunk driving", "dui", "vehicle",
        "road accident", "traffic violation",
        "unfair competition", "antitrust", "cartel",
        "trademark", "imitation", "misleading advertising",
        "extradition", "mutual assistance", "international legal assistance",
        "cultural property", "artifact", "export",
        "parliament", "legislation", "legislative", "motion", "initiative",
        "agriculture", "farming", "subsidy", "direct payment",
        "pharmaceutical", "medicine", "drug approval", "medical device",
        "aviation", "airline", "airport", "flight",
        "railway", "train", "rail", "infrastructure",
        # German
        "strassenverkehr", "verkehrsrecht", "fÃ¼hrerausweis",
        "fÃ¼hrerausweisentzug", "geschwindigkeit", "alkohol",
        "fahrfÃ¤higkeit", "warnungsentzug",
        "unlauterer wettbewerb", "lauterkeit", "nachahmung",
        "verwechslungsgefahr",
        "rechtshilfe", "auslieferung", "spezialitÃ¤tsprinzip",
        "kulturgut", "kulturgÃ¼tertransfer",
        "bundesversammlung", "nationalrat", "stÃ¤nderat",
        "landwirtschaft", "direktzahlungen",
        "arzneimittel", "heilmittel", "swissmedic",
        "luftfahrt", "flugplatz",
        "eisenbahn", "plangenehmigung",
        "svg", "irsg", "uwg", "kgtg", "parlg", "lwg", "hmg", "lfg", "ebg",
    ],
}


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# COURT-SPECIFIC KEYWORDS: Additional triggers for court division selection
# Maps court divisions directly â€” fires when question is about case law / decisions
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_COURT_KEYWORDS: dict[str, list[str]] = {
    # 1B_ â€” Strafprozessuale Zwangsmassnahmen (detention, coercive measures)
    "COURT_STRAFPROZESS": [
        # English
        "pre-trial detention", "pretrial detention", "remand", "detention review",
        "preventive detention", "coercive measure", "seizure order",
        "wiretap order", "surveillance order", "eavesdropping",
        "house search", "search warrant", "unsealing",
        "flight risk", "risk of collusion", "risk of reoffending",
        "detention extension", "release from detention",
        "disproportionate detention", "excessive detention",
        # German
        "untersuchungshaft", "sicherheitshaft", "haftprÃ¼fung",
        "haftverlÃ¤ngerung", "haftentlassung", "Ã¼berhaft",
        "haftgrund", "fluchtgefahr", "kollusionsgefahr", "wiederholungsgefahr",
        "zwangsmassnahme", "beschlagnahme", "Ã¼berwachung", "entsiegelung",
        "hausdurchsuchung", "verhÃ¤ltnismÃ¤ssigkeit der haft",
        "ersatzmassnahme", "kaution", "elektronische fussfessel",
        "1b_",
    ],
    
    # 1C_ â€” Verwaltungsrecht, Raumplanung, Baurecht, Staatshaftung
    "COURT_VERWALTUNG": [
        # English
        "building permit", "zoning", "spatial planning", "construction permit",
        "environmental impact", "expropriation", "state liability",
        "government liability", "concession", "voting rights",
        "political rights", "referendum", "initiative",
        "land use plan", "zone plan",
        # German
        "baubewilligung", "zonenplan", "raumplanung", "nutzungsplanung",
        "umweltvertrÃ¤glichkeit", "enteignung", "staatshaftung",
        "konzession", "stimmrecht", "abstimmung",
        "planungszone", "zonenkonformitÃ¤t", "ausnahmebewilligung",
        "gestaltungsplan", "Ã¼berbauungsordnung",
        "1c_",
    ],
    
    # 2C_ â€” AuslÃ¤nderrecht, Steuerrecht, Gesundheitsrecht
    "COURT_OEFFENTLICH": [
        # English
        "residence permit", "settlement permit", "deportation",
        "family reunification", "hardship case", "integration requirement",
        "naturalization", "citizenship",
        "tax evasion", "tax assessment", "tax ruling",
        "professional license", "medical license", "bar admission",
        "foreigner", "immigrant", "immigration decision",
        "entry ban", "re-entry ban",
        # German
        "aufenthaltsbewilligung", "niederlassungsbewilligung",
        "wegweisung", "familiennachzug", "hÃ¤rtefall",
        "integration", "einbÃ¼rgerung",
        "steuerhinterziehung", "steuerveranlagung", "nachsteuer",
        "berufsausÃ¼bung", "berufsverbot", "bewilligungsentzug",
        "einreiseverbot",
        "2c_", "2d_",
    ],
    
    # 4A_ â€” Vertragsrecht, Haftpflicht, Handelsrecht, Arbeitsrecht
    "COURT_VERTRAG": [
        # English
        "contract interpretation", "contractual liability", "breach of contract",
        "warranty claim", "construction defect", "purchase agreement",
        "employment dispute", "wrongful dismissal", "notice period",
        "corporate dispute", "shareholder", "board of directors",
        "insurance claim", "insurance coverage",
        "commercial lease", "arbitration clause",
        "tort claim", "product liability", "medical malpractice",
        # German
        "vertragsauslegung", "vertragsverletzung", "vertragsbruch",
        "werkvertrag", "kaufvertrag", "auftrag",
        "mangel", "gewÃ¤hrleistung", "nachbesserung",
        "arbeitsvertrag", "kÃ¼ndigung", "kÃ¼ndigungsschutz",
        "arbeitszeugnis", "missbrÃ¤uchliche kÃ¼ndigung",
        "gesellschaftsrecht", "aktionÃ¤r", "verwaltungsrat",
        "versicherungsvertrag", "deckung", "regress",
        "haftpflicht", "produkthaftung", "arzthaftung",
        "4a_", "4d_",
    ],
    
    # 5A_ â€” Familienrecht, Erbrecht, Sachenrecht, PersÃ¶nlichkeitsrecht
    "COURT_FAMILIE": [
        # English
        "divorce", "separation", "alimony", "spousal support",
        "child support", "child custody", "parental authority",
        "visitation rights", "best interest of child",
        "inheritance dispute", "will contest", "forced heirship",
        "disinheritance", "estate partition", "probate",
        "property dispute", "land register", "easement",
        "personality rights", "defamation", "privacy",
        "child protection", "foster care", "guardianship",
        "prenuptial agreement", "matrimonial property",
        # German
        "scheidung", "trennung", "unterhalt", "nachehelicher unterhalt",
        "kindesunterhalt", "sorgerecht", "elterliche sorge",
        "obhut", "kindeswohl", "besuchsrecht", "umgangsrecht",
        "erbstreit", "testament", "pflichtteil", "enterbung",
        "erbteilung", "erbfolge", "vermÃ¤chtnis",
        "grundbuch", "dienstbarkeit", "stockwerkeigentum",
        "persÃ¶nlichkeitsverletzung", "eheschutz",
        "kindesschutz", "beistandschaft", "vormundschaft",
        "gÃ¼terrecht", "errungenschaftsbeteiligung",
        "5a_", "5d_",
    ],
    
    # 6B_ â€” Strafrecht materiell und Strafzumessung
    "COURT_STRAF": [
        # English
        "criminal conviction", "acquittal", "sentencing",
        "conditional sentence", "probation", "parole",
        "expulsion", "criminal expulsion", "hardship clause",
        "assessment of evidence", "evaluation of evidence",
        "measure", "internment", "therapeutic measure",
        "victim rights", "compensation for victim",
        "criminal appeal", "criminal judgment",
        "drug trafficking", "sexual assault conviction",
        "murder conviction", "manslaughter verdict",
        # German
        "schuldspruch", "freispruch", "strafzumessung",
        "bedingte strafe", "unbedingte strafe", "bewÃ¤hrung",
        "landesverweisung", "hÃ¤rtefall",
        "beweiswÃ¼rdigung", "willkÃ¼rliche beweiswÃ¼rdigung",
        "massnahme", "verwahrung", "therapeutische massnahme",
        "opfer", "opferentschÃ¤digung", "genugtuung",
        "strafurteil", "strafmass", "tatkomponenten",
        "tÃ¤terkomponenten", "vorstrafen",
        "6b_", "6f_",
    ],
    
    # 8C_ â€” IV, UV, ALV
    "COURT_SOZIAL_IV": [
        # English
        "disability pension", "disability assessment", "invalidity degree",
        "work capacity assessment", "residual work capacity",
        "income comparison", "tabular wage",
        "accident insurance claim", "occupational disease",
        "natural causation", "adequate causation", "whiplash",
        "unemployment benefit", "unemployment insurance",
        "waiting days", "suitable employment",
        # German
        "invalidenrente", "invaliditÃ¤tsgrad", "arbeitsfÃ¤higkeit",
        "restarbeitsfÃ¤higkeit", "einkommensvergleich", "tabellenlohn",
        "leidensangepasste tÃ¤tigkeit", "iv-stelle", "begutachtung",
        "unfallkausalitÃ¤t", "natÃ¼rliche kausalitÃ¤t", "adÃ¤quanz",
        "schleudertrauma", "integritÃ¤tsentschÃ¤digung",
        "arbeitslosenentschÃ¤digung", "einstelltage",
        "vermittlungsfÃ¤higkeit", "rahmenfrist",
        "8c_", "8d_",
    ],
    
    # 9C_ â€” AHV, KV, EL, BVG
    "COURT_SOZIAL_RENTEN": [
        # English
        "old age pension", "ahv pension", "survivors pension",
        "contribution obligation", "health insurance dispute",
        "health insurance coverage", "supplementary benefits",
        "pension fund", "occupational pension", "vested benefits",
        "premium reduction", "cost sharing",
        # German
        "ahv-rente", "altersrente", "hinterlassenenrente",
        "beitragspflicht", "beitragsjahre",
        "krankenversicherung", "leistungspflicht", "wirtschaftlichkeit",
        "ergÃ¤nzungsleistung", "hilflosenentschÃ¤digung",
        "pensionskasse", "freizÃ¼gigkeit", "austrittsleistung",
        "Ã¼berentschÃ¤digung", "berufliche vorsorge",
        "9c_", "9f_",
    ],
    
    # BGE Leitentscheide â€” criminal/procedural (BGE_IV)
    "COURT_BGE_STRAF": [
        # English
        "leading case", "landmark decision", "precedent", "published decision",
        "federal court precedent", "principle", "fundamental question",
        "bge criminal", "supreme court criminal",
        # German
        "leitentscheid", "grundsatzentscheid", "grundsatzfrage",
        "publizierter entscheid", "bge", "bundesgerichtsentscheid",
        "grundsatz", "praxisÃ¤nderung", "rechtsprechung",
        "bge_iv",
    ],
    
    # BGE Leitentscheide â€” civil (BGE_III)
    "COURT_BGE_ZIVIL": [
        # English
        "leading case civil", "landmark decision civil", "precedent civil",
        "bge civil", "supreme court civil",
        # German
        "leitentscheid zivilrecht", "grundsatzentscheid zivilrecht",
        "bge_iii",
    ],
    
    # BGE Leitentscheide â€” social insurance (BGE_V)
    "COURT_BGE_SOZIAL": [
        # English
        "leading case social", "landmark decision social", "precedent social",
        "bge social", "supreme court social",
        # German
        "leitentscheid sozialversicherung", "grundsatzentscheid sozialversicherung",
        "bge_v",
    ],
    
    # BGE Leitentscheide â€” public law (BGE_I/II)
    "COURT_BGE_OEFFENTLICH": [
        # English
        "leading case public", "landmark decision public", "precedent public",
        "bge public", "supreme court public",
        # German
        "leitentscheid Ã¶ffentliches recht", "grundsatzentscheid verwaltung",
        "bge_i", "bge_ii",
    ],
    
    # 7B_ â€” Bundesstrafgericht (Federal Criminal Court)
    "COURT_BUNDESSTRAF": [
        # English
        "federal criminal court", "federal prosecution", "organized crime",
        "terrorism", "money laundering", "mutual legal assistance",
        "international legal assistance", "federal jurisdiction",
        # German
        "bundesstrafgericht", "beschwerdekammer", "bundesanwaltschaft",
        "bundesgerichtsbarkeit", "organisierte kriminalitÃ¤t",
        "rechtshilfe", "internationale rechtshilfe",
        "geldwÃ¤scherei", "terrorismus", "stbog", "bstkr",
        "7b_",
    ],
}

# Map court keywords â†’ which court sections to include (EXACT keys, no substring matching)
_COURT_KW_TO_SECTIONS: dict[str, list[str]] = {
    "COURT_STRAFPROZESS": ["I. Ã–FFENTLICH-RECHTLICHE ABTEILUNG"],
    "COURT_VERWALTUNG":   ["I. Ã–FFENTLICH-RECHTLICHE ABTEILUNG"],
    "COURT_OEFFENTLICH":  ["II. Ã–FFENTLICH-RECHTLICHE ABTEILUNG"],
    "COURT_VERTRAG":      ["ZIVILRECHTLICHE ABTEILUNGEN"],
    "COURT_FAMILIE":      ["ZIVILRECHTLICHE ABTEILUNGEN"],
    "COURT_STRAF":        ["STRAFRECHTLICHE ABTEILUNG"],
    "COURT_SOZIAL_IV":    ["SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN"],
    "COURT_SOZIAL_RENTEN":["SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN"],
    "COURT_BGE_STRAF":       ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],
    "COURT_BGE_ZIVIL":       ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],
    "COURT_BGE_SOZIAL":      ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],
    "COURT_BGE_OEFFENTLICH": ["BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)"],
    "COURT_BUNDESSTRAF":     ["BUNDESSTRAFGERICHT (Beschwerdekammer & Berufungskammer)"],
}

print(f"\nCourt keyword groups: {len(_COURT_KEYWORDS)} | Total court keywords: {sum(len(v) for v in _COURT_KEYWORDS.values())}")
print(f"Court KWâ†’Section mapping:")
for _ck, _cs in _COURT_KW_TO_SECTIONS.items():
    print(f"  {_ck}: {_cs}")


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# CONTEXT SELECTOR: Score domains, return relevant sections
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def select_planner_context(question: str, max_chars: int = 12000) -> tuple[str, set, dict]:
    """Select relevant routing + court + terminology sections for planner prompt.
    
    Strategy:
    1. Score each LAW domain by keyword hits (multi-word=3pts, word=2pts, substring=1pt)
    2. Score each COURT division by keyword hits (same scoring)
    3. Always include PROZESSRECHT (procedural law appears in every case)
    4. Include top 2 scoring law domains + their mapped court divisions
    5. Include top 1-2 court divisions that scored independently
    6. Always include headers (classification rules) for both laws and courts
    7. Add terminology sections for selected domains
    8. If nothing matches, include Strafrecht + Zivilrecht as broad defaults
    
    Returns: (context_string, selected_domains_set, scores_dict)
    """
    q_lower = question.lower()
    q_words = set(_re.findall(r'[a-zÃ¤Ã¶Ã¼Ã Ã©Ã¨ÃªÃ¯Ã®Ã´Ã¹Ã»Ã§ÃŸ]+', q_lower))
    
    # â”€â”€â”€ Score LAW domains â”€â”€â”€
    scores: dict[str, int] = {}
    for domain, keywords in _DOMAIN_KEYWORDS.items():
        score = 0
        for kw in keywords:
            if " " in kw:
                if kw in q_lower:
                    score += 3
            else:
                if kw in q_words:
                    score += 2
                elif kw in q_lower:
                    score += 1
        scores[domain] = score
    
    # â”€â”€â”€ Score COURT divisions â”€â”€â”€
    court_scores: dict[str, int] = {}
    for court_div, keywords in _COURT_KEYWORDS.items():
        score = 0
        for kw in keywords:
            if " " in kw:
                if kw in q_lower:
                    score += 3
            else:
                if kw in q_words:
                    score += 2
                elif kw in q_lower:
                    score += 1
        court_scores[court_div] = score
    
    # â”€â”€â”€ Select LAW domains: always procedural + top 2 â”€â”€â”€
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    selected_domains = set()
    selected_domains.add("PROZESSRECHT")
    
    non_proc = [(d, s) for d, s in ranked if d != "PROZESSRECHT" and s > 0]
    for domain, _ in non_proc[:2]:
        selected_domains.add(domain)
    
    # If no domain matched, add broad defaults
    if len(selected_domains) <= 1:
        selected_domains.add("STRAFRECHT")
        selected_domains.add("ZIVILRECHT")
    
    # â”€â”€â”€ Select COURT sections â”€â”€â”€
    # Method 1: Mapped from selected law domains
    selected_court_keys: set[str] = set()
    for domain in selected_domains:
        for cdiv in _DOMAIN_TO_COURT_DIVISIONS.get(domain, []):
            for ckey in _COURT_TO_SECTIONS.get(cdiv, []):
                selected_court_keys.add(ckey)
    
    # Method 2: Top scoring court divisions directly (catches court-specific questions)
    ranked_courts = sorted(court_scores.items(), key=lambda x: -x[1])
    top_courts = [(cd, s) for cd, s in ranked_courts if s > 0][:2]
    for court_div, _ in top_courts:
        for ckey in _COURT_KW_TO_SECTIONS.get(court_div, []):
            selected_court_keys.add(ckey)
    
    # â”€â”€â”€ Build context string â”€â”€â”€
    parts = []
    
    # 1. Laws header (classification rules)
    for hkey in _HEADER_KEYS:
        if hkey in _ROUTING_SECTIONS:
            parts.append(_ROUTING_SECTIONS[hkey])
    
    # 2. Laws routing sections for selected domains
    parts.append("\n--- RELEVANTE GESETZES-TAXONOMIE ---\n")
    seen_routing = set()
    for domain in selected_domains:
        config = _DOMAIN_TO_SECTIONS.get(domain, {})
        for rkey in config.get("routing_keys", []):
            if rkey in _ROUTING_SECTIONS and rkey not in seen_routing:
                seen_routing.add(rkey)
                parts.append(_ROUTING_SECTIONS[rkey])
    
    # 3. Court header (classification rules)
    for chkey in _COURT_HEADER_KEYS:
        if chkey in _COURT_SECTIONS:
            parts.append("\n" + _COURT_SECTIONS[chkey])
    
    # 4. Court routing sections for selected divisions
    if selected_court_keys:
        parts.append("\n--- RELEVANTE GERICHTS-ABTEILUNGEN ---\n")
        for ckey in sorted(selected_court_keys):
            if ckey in _COURT_SECTIONS:
                parts.append(_COURT_SECTIONS[ckey])
    
    # 5. Terminology sections for selected domains
    term_parts = []
    seen_term = set()
    for domain in selected_domains:
        config = _DOMAIN_TO_SECTIONS.get(domain, {})
        for tkey in config.get("terminology_keys", []):
            if tkey in _TERMINOLOGY_SECTIONS and tkey not in seen_term:
                seen_term.add(tkey)
                term_parts.append(_TERMINOLOGY_SECTIONS[tkey])
    
    if term_parts:
        parts.append("\n--- TERMINOLOGIE (Englisch â†’ Deutsch) ---\n")
        parts.extend(term_parts)
    
    # 6. Always include application rules (short, universally useful)
    _app_keys = _find_section_keys(_TERMINOLOGY_SECTIONS, ["anwendungsregel"])
    for akey in _app_keys:
        if akey in _TERMINOLOGY_SECTIONS and akey not in seen_term:
            parts.append("\n" + _TERMINOLOGY_SECTIONS[akey])
    
    context = "\n\n".join(parts)
    
    # Safety truncation
    if len(context) > max_chars:
        context = context[:max_chars] + "\n[... gekÃ¼rzt ...]"
    
    return context, selected_domains, {**scores, **court_scores}


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# PLANNER FUNCTION
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def run_planner(question: str) -> Optional[Plan]:
    """Run Planner LLM to decompose question into research directions."""
    # 1. Select relevant context
    context_text, selected_domains, scores = select_planner_context(question)
    
    # 2. System prompt
    system_prompt = planner_system_template.format(
        available_law_codes=LAW_TYPES_FOR_PROMPT,
        available_court_codes=COURT_TYPES_FOR_PROMPT,
    )
    
    # 3. User message
    user_msg = f"{context_text}\n\nFRAGE: {question}"
    
    # 4. LLM call
    prompt = f"[INST] {system_prompt}\n\n{user_msg} [/INST]"
    response = llm(
        prompt,
        max_tokens=CONFIG["max_tokens_planner"],
        temperature=CONFIG["temperature"],
        grammar=planner_grammar,
        stop=["[INST]", "</s>"],
    )
    raw = response["choices"][0]["text"].strip()
    
    # Parse JSON (with retry)
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        # Guard: retry only if appended prompt fits in context window
        prompt2 = prompt + "\n" + raw + "\n[INST] Output NUR valides JSON. [/INST]"
        if len(prompt2) // 3 > CONFIG["n_ctx"] - CONFIG["max_tokens_planner"] - 200:
            return None  # Would overflow context â€” skip to fallback
        response2 = llm(prompt2, max_tokens=CONFIG["max_tokens_planner"],
                        temperature=0.0, grammar=planner_grammar, stop=["[INST]", "</s>"])
        raw2 = response2["choices"][0]["text"].strip()
        try:
            data = json.loads(raw2)
        except json.JSONDecodeError:
            return None
    
    # Build Plan
    directions = []
    for d in data.get("directions", []):
        valid_codes = [c for c in d.get("filter_codes", [])
                       if c in law_code_to_indices or c in court_code_to_indices]
        directions.append(Direction(
            priority=d.get("priority", 50),
            corpus=d.get("corpus", "laws"),
            rechtsgebiet=d.get("rechtsgebiet", ""),
            filter_codes=valid_codes,
            reasoning=d.get("reasoning", ""),
            seed_queries=d.get("seed_queries", []),
        ))
    

    # --- POST-PROCESSING: Deduplicate filters + Enrich single-code directions ---
    # Related codes: if direction has only 1 filter, add its natural companions
    _RELATED_CODES = {
        "StPO": ["BStKR", "JStPO"],
        "StGB": ["JStG"],
        "OR": ["ZGB"],
        "ZGB": ["OR"],
        "BGG": ["BV"],
        "BV": ["BGG"],
        "IVG": ["ATSG"],
        "ATSG": ["IVG", "IVV"],
        "AIG": ["BV"],
        "DBG": ["StHG"],
        "StHG": ["DBG"],
        "1B_": ["7B_"],
        "7B_": ["1B_"],
        "6B_": ["BGE_IV"],
        "BGE_IV": ["6B_"],
        "8C_": ["9C_", "BGE_V"],
        "9C_": ["8C_", "BGE_V"],
        "BGE_V": ["8C_", "9C_"],
        "4A_": ["4D_", "BGE_III"],
        "5A_": ["BGE_III"],
        "2C_": ["BGE_I", "BGE_II"],
    }
    
    # Rechtsgebiet -> fallback codes when dedup strips everything
    _DOMAIN_FALLBACK = {
        "strafprozess": ["StPO", "BStKR", "JStPO"],
        "strafrecht": ["StGB", "JStG"],
        "zivilrecht": ["OR", "ZGB"],
        "familienrecht": ["ZGB", "OR"],
        "prozessrecht": ["BGG", "BV"],
        "verfahrensrecht": ["BGG", "BV", "VwVG"],
        "sozialversicherung": ["IVG", "ATSG", "IVV"],
        "iv": ["IVG", "ATSG"],
        "oeffentliches_recht": ["AIG", "BV", "VwVG"],
        "steuerrecht": ["DBG", "StHG"],
        "finanzmarktrecht": ["FIDLEG", "FINMAG"],
        "strafverfahren": ["StPO", "BStKR"],
        "leitentscheide": ["BGE_I", "BGE_II", "BGE_III", "BGE_IV", "BGE_V"],
    }

    used_codes = set()
    for direction in directions:
        # Deduplicate: remove codes already used by higher-priority directions
        original = direction.filter_codes[:]
        direction.filter_codes = [c for c in direction.filter_codes if c not in used_codes]
        
        # Enrich: if 0 or 1 codes remain after dedup
        if len(direction.filter_codes) == 0:
            # All codes stripped -> assign from rechtsgebiet
            rg = direction.rechtsgebiet.lower().replace("-", "").replace(" ", "_")
            # Try matching domain keys
            fallback = []
            for key, codes in _DOMAIN_FALLBACK.items():
                if key in rg or rg in key:
                    fallback = codes
                    break
            # Add unused fallback codes
            for fc in fallback:
                if fc not in used_codes:
                    if fc in law_code_to_indices or fc in court_code_to_indices:
                        direction.filter_codes.append(fc)
                if len(direction.filter_codes) >= 3:
                    break
        
        if len(direction.filter_codes) == 1:
            # Single code -> add related companions
            base = direction.filter_codes[0]
            for related in _RELATED_CODES.get(base, []):
                if related not in used_codes and related not in direction.filter_codes:
                    if related in law_code_to_indices or related in court_code_to_indices:
                        direction.filter_codes.append(related)
        
        # Track all codes this direction now owns
        used_codes.update(direction.filter_codes)
    # --- END POST-PROCESSING ---

    return Plan(
        sachverhalt=data.get("sachverhalt", ""),
        rechtsfragen=data.get("rechtsfragen", []),
        directions=sorted(directions, key=lambda d: d.priority),
    )


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# FALLBACK (when planner JSON parsing fails)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def fallback_decompose(question: str) -> Plan:
    """Keyword-based fallback when planner fails â€” uses same scoring logic."""
    _, selected_domains, scores = select_planner_context(question)
    
    _DOMAIN_DEFAULT_CODES = {
        "STRAFRECHT": [("laws", ["StGB"]), ("courts", ["6B_", "BGE_IV"])],
        "STRAFPROZESS": [("laws", ["StPO", "StBOG"]), ("courts", ["1B_", "BGE_IV", "7B_"])],
        "ZIVILRECHT": [("laws", ["OR", "ZGB"]), ("courts", ["4A_", "5A_", "BGE_III"])],
        "PROZESSRECHT": [("laws", ["BGG", "BV"]), ("courts", ["BGE_I"])],
        "SOZIALVERSICHERUNG": [("laws", ["IVG", "ATSG"]), ("courts", ["8C_", "9C_", "BGE_V"])],
        "OEFFENTLICHES_RECHT": [("laws", ["AIG", "BV"]), ("courts", ["2C_", "BGE_I"])],
        "STEUERRECHT": [("laws", ["DBG"]), ("courts", ["2C_", "BGE_II"])],
        "FINANZMARKTRECHT": [("laws", ["FIDLEG", "FINMAG"]), ("courts", ["2C_"])],
        "WEITERE": [("laws", ["SVG"]), ("courts", ["6B_", "1C_", "7B_"])],
    }
    
    directions = []
    priority = 1
    for domain in selected_domains:
        for corpus, codes in _DOMAIN_DEFAULT_CODES.get(domain, []):
            valid_codes = [c for c in codes if c in law_code_to_indices or c in court_code_to_indices]
            if valid_codes:
                directions.append(Direction(
                    priority=priority, corpus=corpus,
                    rechtsgebiet=domain, filter_codes=valid_codes,
                    reasoning=f"Domain match: {domain} (score={scores.get(domain, 0)})",
                    seed_queries=[],
                ))
                priority += 1
    
    # Always procedural
    directions.append(Direction(
        priority=99, corpus="laws", rechtsgebiet="Verfahrensrecht",
        filter_codes=["BGG", "BV"],
        reasoning="Procedural defaults",
        seed_queries=["Beschwerde Bundesgericht Legitimation Frist"],
    ))
    
    if len(directions) <= 1:
        directions.insert(0, Direction(priority=1, corpus="laws", rechtsgebiet="",
                                       filter_codes=[], reasoning="Broad search",
                                       seed_queries=[]))
        directions.insert(1, Direction(priority=2, corpus="courts", rechtsgebiet="",
                                       filter_codes=[], reasoning="Broad court search",
                                       seed_queries=[]))
    
    return Plan(sachverhalt="", rechtsfragen=[], directions=sorted(directions, key=lambda d: d.priority))


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# VALIDATION: Test with multiple question types to verify coverage
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

_test_questions = [
    "Under what conditions can pre-trial detention be extended?",
    "What are the requirements for a valid will in Switzerland?",
    "How is the disability pension calculated for partial invalidity?",
    "What constitutes unfair competition under Swiss law?",
    "Can a foreigner appeal a deportation order?",
    "What are the tax implications of capital gains?",
    "What are the requirements for bank licensing in Switzerland?",
    "What is the standard for evaluating evidence in a criminal conviction?",
    "How is child support calculated after divorce?",
]

print(f"\nPlanner ready (keyword-based dynamic context selection)")
print(f"  Law domains: {len(_DOMAIN_KEYWORDS)} | Law keywords: {sum(len(v) for v in _DOMAIN_KEYWORDS.values())}")
print(f"  Court divisions: {len(_COURT_KEYWORDS)} | Court keywords: {sum(len(v) for v in _COURT_KEYWORDS.values())}")
print(f"\n{'='*70}")
print("COVERAGE TEST â€” verifying all domains + courts are reachable:")
print(f"{'='*70}")
for _tq in _test_questions:
    _ctx, _sel, _sc = select_planner_context(_tq)
    _law_scores = {k: v for k, v in _sc.items() if not k.startswith("COURT_")}
    _court_scores = {k: v for k, v in _sc.items() if k.startswith("COURT_")}
    _top_law = sorted(_law_scores.items(), key=lambda x: -x[1])[:2]
    _top_court = sorted(_court_scores.items(), key=lambda x: -x[1])[:2]
    print(f"\n  Q: {_tq[:70]}")
    print(f"  â†’ Law domains: {_sel} | Context: {len(_ctx):,} chars (~{len(_ctx)//4} tok)")
    print(f"  â†’ Top law:   {_top_law}")
    print(f"  â†’ Top court: {_top_court}")


Laws routing sections: ['GESETZES-ROUTING (search_laws)', 'STRAFRECHT (materiell)', 'STRAFPROZESSRECHT', 'ZIVILRECHT', 'PROZESSRECHT', 'SOZIALVERSICHERUNGSRECHT', 'ÖFFENTLICHES RECHT', 'STEUERRECHT', 'FINANZMARKTRECHT', 'WEITERE WICHTIGE GESETZE']
Court routing sections: ['GERICHTS-ROUTING (search_courts)', 'I. ÖFFENTLICH-RECHTLICHE ABTEILUNG', 'II. ÖFFENTLICH-RECHTLICHE ABTEILUNG', 'ZIVILRECHTLICHE ABTEILUNGEN', 'STRAFRECHTLICHE ABTEILUNG', 'SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN', 'BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)', 'BUNDESSTRAFGERICHT (Beschwerdekammer & Berufungskammer)']
  Court sizes: {'GERICHTS-ROUTING (search_courts)': 1129, 'I. ÖFFENTLICH-RECHTLICHE ABTEILUNG': 1079, 'II. ÖFFENTLICH-RECHTLICHE ABTEILUNG': 556, 'ZIVILRECHTLICHE ABTEILUNGEN': 1288, 'STRAFRECHTLICHE ABTEILUNG': 657, 'SOZIALVERSICHERUNGSRECHTLICHE ABTEILUNGEN': 1124, 'BGE LEITENTSCHEIDE (Publizierte Grundsatzentscheide)': 2760, 'BUNDESSTRAFGERICHT (Beschwerdekammer & Berufungskammer)': 2282}

In [ ]:
# Cell 14: Direction Executor (ReAct Loop)

def format_observation(results: list[tuple[str, float, str]], max_items: int = 10) -> str:
    """Format search results for executor observation (scores + snippets)."""
    lines = []
    for i, (cit, score, snippet) in enumerate(results[:max_items]):
        lines.append(f"{i+1}. {cit} ({score:.2f}): {snippet[:80]}")
    return "\n".join(lines)


def format_prior_findings(citations: list[tuple[str, float, str]], top_with_snippets: int = 5, max_items: int = 20) -> str:
    """Format prior findings for context injection.
    Shows snippets for the top N entries (by score), citation+score only for the rest."""
    if not citations:
        return "Noch keine Funde aus vorherigen Richtungen."
    # Sort by score descending, take last max_items
    recent = citations[-max_items:]
    sorted_by_score = sorted(recent, key=lambda x: x[1], reverse=True)
    lines = ["BISHERIGE FUNDE (beste zuerst):"]
    for i, (cit, score, snippet) in enumerate(sorted_by_score):
        if i < top_with_snippets and snippet:
            lines.append(f"- {cit} ({score:.2f}): {snippet[:80]}")
        else:
            lines.append(f"- {cit} ({score:.2f})")
    return "\n".join(lines)


def format_direction_history(history: list[dict]) -> str:
    """Format this direction's search history with scores + snippets."""
    if not history:
        return "Noch keine Suchen in dieser Richtung."
    lines = []
    for entry in history:
        query = entry["query"]
        lines.append(f"Query: '{query}'")
        for cit, score, snippet in entry.get("results", [])[:5]:
            lines.append(f"  {cit} ({score:.2f}): {snippet[:80]}")
        if not entry.get("results"):
            # Legacy fallback if results not stored
            cits = "; ".join(f"{c} ({s:.2f})" for c, s in entry.get("citations", [])[:5])
            lines.append(f"  â†’ {cits}")
    return "\n".join(lines)


def run_direction(
    question: str,
    plan: Plan,
    direction: Direction,
    direction_idx: int,
    prior_findings: list[tuple[str, float, str]],
) -> list[tuple[str, float, str]]:
    """Execute one research direction with ReAct loop.
    Returns list of (citation, score, snippet) tuples."""
    start_time = time.time()
    direction_citations: list[tuple[str, float, str]] = []
    history: list[dict] = []

    # --- Iteration 0: Seed query (NO LLM call) ---
    if direction.seed_queries:
        seed_q = direction.seed_queries[0]
        results = filtered_hybrid_search(seed_q, direction.corpus, direction.filter_codes,
                                          top_k=CONFIG["search_top_k"])
        direction_citations.extend(results)
        history.append({"query": seed_q, "results": results})

    # --- Iterations 1-3: ReAct LLM loop ---
    # Select prompt template
    is_procedural = direction.priority >= 90
    plan_summary = (
        f"Sachverhalt: {plan.sachverhalt}\n"
        f"Rechtsfragen: {'; '.join(plan.rechtsfragen)}\n"
        f"Dies ist Richtung {direction_idx+1} von {len(plan.directions)}."
    )

    for iteration in range(1, CONFIG["max_executor_iterations"] + 1):
        # Timeout check
        if time.time() - start_time > CONFIG["executor_timeout_sec"]:
            break

        # Build prompt
        if is_procedural:
            system_content = executor_procedural_template.format(
                prior_findings=format_prior_findings(prior_findings + direction_citations),
            )
        else:
            taxonomy_section = get_taxonomy_section(direction.filter_codes)
            system_content = executor_system_template.format(
                rechtsgebiet=direction.rechtsgebiet,
                corpus=direction.corpus,
                filter_codes=", ".join(direction.filter_codes),
                reasoning=direction.reasoning,
                taxonomy_section=taxonomy_section,
                plan_summary=plan_summary,
                prior_findings=format_prior_findings(prior_findings + direction_citations),
                direction_history=format_direction_history(history),
            )

        prompt = f"[INST] {system_content}\n\nGeneriere deine nÃ¤chste Suchanfrage oder signalisiere done. [/INST]"
        response = llm(
            prompt,
            max_tokens=CONFIG["max_tokens_executor"],
            temperature=CONFIG.get("temperature_executor", 0.3),
            grammar=executor_grammar,
            stop=["[INST]", "</s>"],
        )
        raw = response["choices"][0]["text"].strip()

        # Parse response
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            break  # Can't parse â†’ stop this direction

        # Check done
        if parsed.get("done", False) or not parsed.get("query", "").strip():
            break

        # Check for repeated query (exact match)
        query = parsed["query"].strip()
        if query in [h["query"] for h in history]:
            break

        # --- EMBEDDING DIVERSITY CHECK ---
        # Reject queries that are semantically too similar to previous ones
        # (catches synonym rephrasing that word-overlap rules miss)
        if history:
            new_emb = embed_query(query, direction.corpus)  # (1, dim)
            too_similar = False
            for prev in history:
                prev_emb = embed_query(prev["query"], direction.corpus)
                sim = float((new_emb @ prev_emb.T)[0, 0])
                if sim > 0.80:
                    too_similar = True
                    break
            if too_similar:
                break  # Query too similar to a previous one â€” stop direction


        # Execute search
        results = filtered_hybrid_search(query, direction.corpus, direction.filter_codes,
                                          top_k=CONFIG["search_top_k"])

        # Adaptive fallback: if 0 results with filter, try unfiltered
        if not results and direction.filter_codes:
            results = filtered_hybrid_search(query, direction.corpus, [], top_k=CONFIG["search_top_k"])

        direction_citations.extend(results)
        history.append({"query": query, "results": results})

    return direction_citations


print("Executor ready.")

Executor ready.


In [ ]:
# Cell 15: Procedural Defaults + Aggregation

UNIVERSAL_DEFAULTS = [
    "Art. 42 Abs. 2 BGG", "Art. 95 BGG", "Art. 100 Abs. 1 BGG",
    "Art. 105 Abs. 1 BGG", "Art. 29 Abs. 2 BV",
]

CASE_TYPE_DEFAULTS = {
    "criminal": ["Art. 78 Abs. 1 BGG", "Art. 80 Abs. 1 BGG", "Art. 81 Abs. 1 BGG"],
    "civil": ["Art. 72 Abs. 1 BGG", "Art. 74 Abs. 1 BGG", "Art. 76 Abs. 1 BGG"],
    "public_law": ["Art. 82 BGG", "Art. 89 Abs. 1 BGG"],
    "social_insurance": ["Art. 61 ATSG", "Art. 16 ATSG"],
}

SUBTYPE_DEFAULTS = {
    "1B_": ["Art. 221 Abs. 1 StPO", "Art. 10 Abs. 2 BV", "Art. 31 Abs. 3 BV"],
    "5A_": ["Art. 98 BGG", "Art. 9 BV"],
    "6B_": ["Art. 47 StGB", "Art. 50 StGB"],
    "8C_": ["Art. 4 Abs. 1 IVG", "Art. 16 ATSG"],
}


def detect_case_type(citations: list[tuple[str, float, str]]) -> tuple[str, list[str]]:
    """Detect case type from found citations. Returns (type, detected_prefixes)."""
    prefixes = set()
    for cit, _, *_rest in citations:
        m = _COURT_PREFIX_RE.match(cit)
        if m:
            prefixes.add(m.group(1))
        elif cit.startswith("BGE"):
            bm = _BGE_RE.match(cit)
            if bm:
                prefixes.add(f"BGE_{bm.group(2)}")

    if "6B_" in prefixes or "1B_" in prefixes:
        return "criminal", list(prefixes)
    elif "4A_" in prefixes or "5A_" in prefixes:
        return "civil", list(prefixes)
    elif "8C_" in prefixes or "9C_" in prefixes:
        return "social_insurance", list(prefixes)
    elif "2C_" in prefixes or "1C_" in prefixes:
        return "public_law", list(prefixes)
    return "unknown", list(prefixes)


def get_procedural_defaults(case_type: str, prefixes: list[str]) -> list[str]:
    """Get procedural default citations that should always be included."""
    defaults = list(UNIVERSAL_DEFAULTS)
    if case_type in CASE_TYPE_DEFAULTS:
        defaults.extend(CASE_TYPE_DEFAULTS[case_type])
    for prefix in prefixes:
        if prefix in SUBTYPE_DEFAULTS:
            defaults.extend(SUBTYPE_DEFAULTS[prefix])
    # Only return those that actually exist in corpus
    return [d for d in defaults if d in corpus_citation_set]


# Regex for extracting explicit citations from question text
_CITATION_RE = re.compile(
    r"(?:Art\.?\s*\d+[\w\s.]{0,20}?\b(?:BGG|BV|OR|ZGB|StGB|StPO|ATSG|IVG|SchKG|ZPO|AIG|KVG|UVG)\b)"
    r"|(?:BGE\s+\d+\s+I{1,3}V?\s+\d+)"
    r"|(?:\d[A-Z]_\d+/\d+)",
)


def aggregate_and_output(
    all_direction_citations: list[list[tuple[str, float, str]]],
    question: str,
    rerank_query: str = "",
    return_diagnostics: bool = False,
) -> str | tuple[str, dict]:
    """Phase 3: Aggregate, inject defaults, rerank, format output.
    rerank_query: German text for reranker (sachverhalt). Falls back to question if empty.
    If return_diagnostics=True, returns (result_string, diagnostics_dict)."""
    # Use German sachverhalt for reranking if available, else original question
    _rerank_q = rerank_query if rerank_query else question
    
    # 1. Flatten
    all_citations: list[tuple[str, float]] = []
    for direction_cits in all_direction_citations:
        all_citations.extend((cit, score) for cit, score, *_ in direction_cits)

    # 2. Inject procedural defaults
    case_type, prefixes = detect_case_type(all_citations)
    defaults = get_procedural_defaults(case_type, prefixes)
    # --- SMART DEFAULT FILTERING (cosine similarity) ---
    # Only inject defaults that are semantically relevant to this query.
    # Prevents 8-9 irrelevant boilerplate citations from polluting output.
    if defaults:
        query_emb = embed_query(_rerank_q, "laws")  # (1, dim)
        default_texts = [citation_to_text.get(d, d) for d in defaults]
        default_embs = _st_model.encode(
            default_texts,
            prompt=CONFIG["prompt_doc_law"],
            normalize_embeddings=True,
            batch_size=len(defaults),
        ).astype("float32")
        sims = (query_emb @ default_embs.T)[0]  # shape (n_defaults,)
        DEFAULTS_SIM_THRESHOLD = 0.30
        for default_cit, sim in zip(defaults, sims):
            if sim >= DEFAULTS_SIM_THRESHOLD:
                all_citations.append((default_cit, float(sim) * 0.005))  # Tiny score so real results still rank above
        defaults_injected = [d for d, s in zip(defaults, sims) if s >= DEFAULTS_SIM_THRESHOLD]
    else:
        defaults_injected = []

    # 3. Deduplicate (keep highest score)
    citation_scores: dict[str, float] = {}
    for cit, score in all_citations:
        if cit not in citation_scores or score > citation_scores[cit]:
            citation_scores[cit] = score

    if not citation_scores:
        if return_diagnostics:
            return "", {"scored_all": [], "candidates": [], "rerank_time": 0.0,
                        "case_type": case_type, "prefixes": prefixes, "defaults": defaults_injected if "defaults_injected" in dir() else defaults}
        return ""

    # 4. RERANKER BYPASSED â€” Qwen3-Reranker produces uniform scores (~0.0097),
    #    nothing passes 0.2 cutoff â†’ always falls back to top-10.
    #    Instead, use RRF fusion scores directly (already ranked by hybrid search quality).
    candidates = list(citation_scores.keys())
    t_rerank = time.time()
    # --- POST-RETRIEVAL QUALITY FILTER (cosine similarity) ---
    # Score each candidate's document text against the query.
    # This replaces the broken Qwen3-Reranker with a working semantic filter.
    if len(citation_scores) > 5:  # Only filter if we have enough candidates
        query_emb = embed_query(_rerank_q, "laws")  # (1, dim)
        cit_list = list(citation_scores.keys())
        doc_texts = [citation_to_text.get(c, c)[:512] for c in cit_list]
        doc_embs = _st_model.encode(
            doc_texts,
            prompt=CONFIG["prompt_doc_law"],
            normalize_embeddings=True,
            batch_size=CONFIG["embed_batch_size"],
        ).astype("float32")
        doc_sims = (query_emb @ doc_embs.T)[0]  # shape (n_candidates,)
        POST_RETRIEVAL_SIM_FLOOR = 0.20  # Drop clearly irrelevant results
        for cit, sim in zip(cit_list, doc_sims):
            if sim < POST_RETRIEVAL_SIM_FLOOR and citation_scores[cit] < 0.01:
                # Only drop if BOTH RRF score is low AND embedding similarity is low
                del citation_scores[cit]

    # Sort by RRF score descending (higher = better retrieval match)
    scored = sorted(citation_scores.items(), key=lambda x: x[1], reverse=True)
    rerank_time = time.time() - t_rerank

    # 5. Cap at max_final_citations (no cutoff needed â€” RRF scores are meaningful)
    final = scored[:CONFIG["max_final_citations"]]

    # 6. Prepend regex-extracted explicit citations
    explicit = _CITATION_RE.findall(question)
    for cit in reversed(explicit):
        cit = cit.strip()
        if cit in corpus_citation_set and cit not in [f[0] for f in final]:
            final.insert(0, (cit, 1.0))

    result = ";".join(cit for cit, _ in final)

    if return_diagnostics:
        diag = {
            "scored_all": scored,  # All candidates with reranker scores (sorted desc)
            "candidates": candidates,
            "citation_scores_rrf": citation_scores,  # RRF scores before reranking
            "rerank_time": rerank_time,
            "case_type": case_type,
            "prefixes": prefixes,
            "defaults": defaults,
            "explicit": explicit,
            "no_text": [c for c in candidates if c not in citation_to_text],
        }
        return result, diag
    return result


print("Aggregation ready.")
print(f"  Universal defaults: {len(UNIVERSAL_DEFAULTS)}")
print(f"  Case type defaults: {sum(len(v) for v in CASE_TYPE_DEFAULTS.values())} across {len(CASE_TYPE_DEFAULTS)} types")

Aggregation ready.
  Universal defaults: 5
  Case type defaults: 10 across 4 types


In [ ]:
# Cell 16: Full Pipeline Orchestrator

def run_pipeline(question: str, verbose: bool = False) -> str:
    """Run the full Planner-Director pipeline on a single question."""
    t_start = time.time()

    # â•â•â• PHASE 1: PLANNER â•â•â•
    if verbose:
        print(f"\n{'='*60}")
        print(f"QUESTION: {question[:100]}...")
        print(f"{'='*60}")
        print("\n[PHASE 1] Planning...")

    plan = run_planner(question)
    if plan is None:
        if verbose:
            print("  Planner failed â†’ using fallback decomposition")
        plan = fallback_decompose(question)

    # Validate: ensure min 3 directions
    while len(plan.directions) < 3:
        # Add catch-all unfiltered
        plan.directions.insert(0, Direction(
            priority=50, corpus="laws" if len(plan.directions) % 2 == 0 else "courts",
            rechtsgebiet="Allgemein",
            filter_codes=[], reasoning="Catch-all broad search",
            seed_queries=[plan.sachverhalt[:80]] if plan.sachverhalt else [],
        ))

    if verbose:
        print(f"  Plan: {len(plan.directions)} directions")
        for i, d in enumerate(plan.directions):
            print(f"    Dir {i+1} (P{d.priority}): {d.corpus}/{d.filter_codes} â€” {d.reasoning[:60]}")

    # â•â•â• PHASE 2: DIRECTION EXECUTORS â•â•â•
    if verbose:
        print(f"\n[PHASE 2] Executing {len(plan.directions)} directions...")

    all_direction_citations: list[list[tuple[str, float, str]]] = []
    prior_findings: list[tuple[str, float, str]] = []

    for i, direction in enumerate(plan.directions):
        t_dir = time.time()
        direction_cits = run_direction(question, plan, direction, i, prior_findings)

        if verbose:
            print(f"    Dir {i+1}: {len(direction_cits)} citations ({time.time()-t_dir:.1f}s)")

        all_direction_citations.append(direction_cits)
        # Update prior findings (rolling window of last 20)
        prior_findings = (prior_findings + direction_cits)[-20:]

    # â•â•â• PHASE 3: AGGREGATION â•â•â•
    if verbose:
        print(f"\n[PHASE 3] Aggregating + reranking...")

    result = aggregate_and_output(all_direction_citations, question, rerank_query=plan.sachverhalt)

    if verbose:
        n_final = len(result.split(";")) if result else 0
        print(f"  Final: {n_final} citations")
        print(f"  Total time: {time.time()-t_start:.1f}s")

    return result


print("Pipeline ready.")

Pipeline ready.


In [ ]:
# Cell 16.5: Comprehensive Pipeline Logger
# Creates a detailed log file capturing EVERY step of the pipeline execution.
# Log file: OUTPUT_PATH / "pipeline_debug_log.txt"

import logging
from datetime import datetime

# â”€â”€â”€ Setup file logger â”€â”€â”€
LOG_PATH = OUTPUT_PATH / "pipeline_debug_log.txt"
_pipeline_logger = logging.getLogger("pipeline_debug")
_pipeline_logger.setLevel(logging.DEBUG)
_pipeline_logger.handlers.clear()  # Reset on re-run

_file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_file_handler.setLevel(logging.DEBUG)
_file_handler.setFormatter(logging.Formatter("%(message)s"))
_pipeline_logger.addHandler(_file_handler)

# Also stream to stdout for visibility
_stream_handler = logging.StreamHandler()
_stream_handler.setLevel(logging.INFO)
_stream_handler.setFormatter(logging.Formatter("%(message)s"))
_pipeline_logger.addHandler(_stream_handler)

L = _pipeline_logger  # shorthand


def _log_separator(char="â•", width=80):
    L.debug(char * width)


def _log_header(title: str, level: int = 1):
    if level == 1:
        _log_separator("â•")
        L.debug(f"  {title}")
        _log_separator("â•")
    elif level == 2:
        _log_separator("â”€", 60)
        L.debug(f"  {title}")
        _log_separator("â”€", 60)
    else:
        L.debug(f"  â–¸ {title}")


# â”€â”€â”€ Instrumented pipeline functions â”€â”€â”€

def run_planner_logged(question: str) -> Optional[Plan]:
    """Instrumented run_planner with full logging."""
    _log_header("PHASE 1: PLANNER", 1)
    L.debug(f"  Input question: {question}")
    L.debug("")
    
    # 1. Context selection
    _log_header("Context Selection (keyword matching)", 2)
    context_text, selected_domains, scores = select_planner_context(question)
    
    # Log all scores
    law_scores = {k: v for k, v in scores.items() if not k.startswith("COURT_")}
    court_scores = {k: v for k, v in scores.items() if k.startswith("COURT_")}
    
    L.debug("  LAW DOMAIN SCORES:")
    for domain, score in sorted(law_scores.items(), key=lambda x: -x[1]):
        marker = " â˜…" if domain in selected_domains else ""
        L.debug(f"    {domain:25s} â†’ {score:3d}{marker}")
    
    L.debug("  COURT DIVISION SCORES:")
    for court_div, score in sorted(court_scores.items(), key=lambda x: -x[1]):
        L.debug(f"    {court_div:25s} â†’ {score:3d}")
    
    L.debug(f"\n  Selected domains: {selected_domains}")
    L.debug(f"  Context length: {len(context_text):,} chars (~{len(context_text)//4} tokens)")
    L.debug(f"\n  --- FULL INJECTED CONTEXT ---")
    L.debug(context_text)
    L.debug(f"  --- END INJECTED CONTEXT ---")
    L.debug("")
    
    # 2. System prompt
    system_prompt = planner_system_template.format(
        available_law_codes=LAW_TYPES_FOR_PROMPT,
        available_court_codes=COURT_TYPES_FOR_PROMPT,
    )
    L.debug(f"  System prompt length: {len(system_prompt):,} chars (~{len(system_prompt)//4} tokens)")
    L.debug(f"\n  --- FULL SYSTEM PROMPT ---")
    L.debug(system_prompt)
    L.debug(f"  --- END SYSTEM PROMPT ---\n")
    
    # 3. User message
    user_msg = f"{context_text}\n\nFRAGE: {question}"
    L.debug(f"  User message length: {len(user_msg):,} chars (~{len(user_msg)//4} tokens)")
    
    # 4. Full prompt
    prompt = f"[INST] {system_prompt}\n\n{user_msg} [/INST]"
    total_chars = len(prompt)
    L.debug(f"  TOTAL PROMPT: {total_chars:,} chars (~{total_chars//4} tokens)")
    L.debug("")
    
    _log_header("LLM Call (Planner)", 2)
    t0 = time.time()
    response = llm(
        prompt,
        max_tokens=CONFIG["max_tokens_planner"],
        temperature=CONFIG["temperature"],
        grammar=planner_grammar,
        stop=["[INST]", "</s>"],
    )
    elapsed = time.time() - t0
    raw = response["choices"][0]["text"].strip()
    
    L.debug(f"  LLM response time: {elapsed:.2f}s")
    L.debug(f"  RAW LLM OUTPUT:")
    L.debug(f"  {raw}")
    L.debug("")
    
    # 5. Parse
    _log_header("Planner JSON Parsing", 2)
    try:
        data = json.loads(raw)
        L.debug("  âœ“ JSON parsed successfully")
    except json.JSONDecodeError as e:
        L.debug(f"  âœ— JSON parse error: {e}")
        L.debug("  Attempting retry with correction prompt...")
        prompt2 = prompt + "\n" + raw + "\n[INST] Output NUR valides JSON. [/INST]"
        if len(prompt2) // 3 > CONFIG["n_ctx"] - CONFIG["max_tokens_planner"] - 200:
            L.debug("  âœ— Retry would overflow context window â€” returning None")
            return None
        response2 = llm(prompt2, max_tokens=CONFIG["max_tokens_planner"],
                        temperature=0.0, grammar=planner_grammar, stop=["[INST]", "</s>"])
        raw2 = response2["choices"][0]["text"].strip()
        L.debug(f"  Retry raw output: {raw2}")
        try:
            data = json.loads(raw2)
            L.debug("  âœ“ Retry JSON parsed successfully")
        except json.JSONDecodeError:
            L.debug("  âœ— Retry also failed â†’ returning None (will use fallback)")
            return None
    
    # 6. Build Plan
    _log_header("Plan Construction", 2)
    L.debug(f"  Sachverhalt: {data.get('sachverhalt', '')}")
    L.debug(f"  Rechtsfragen: {data.get('rechtsfragen', [])}")
    
    directions = []
    for i, d in enumerate(data.get("directions", [])):
        raw_codes = d.get("filter_codes", [])
        valid_codes = [c for c in raw_codes
                       if c in law_code_to_indices or c in court_code_to_indices]
        invalid_codes = [c for c in raw_codes if c not in valid_codes]
        
        direction = Direction(
            priority=d.get("priority", 50),
            corpus=d.get("corpus", "laws"),
            rechtsgebiet=d.get("rechtsgebiet", ""),
            filter_codes=valid_codes,
            reasoning=d.get("reasoning", ""),
            seed_queries=d.get("seed_queries", []),
        )
        directions.append(direction)
        
        L.debug(f"\n  Direction {i+1}:")
        L.debug(f"    Priority: {direction.priority}")
        L.debug(f"    Corpus: {direction.corpus}")
        L.debug(f"    Rechtsgebiet: {direction.rechtsgebiet}")
        L.debug(f"    Filter codes: {valid_codes}")
        if invalid_codes:
            L.debug(f"    INVALID codes (dropped): {invalid_codes}")
        L.debug(f"    Reasoning: {direction.reasoning}")
        L.debug(f"    Seed queries: {direction.seed_queries}")
    

    # --- POST-PROCESSING: Deduplicate filters + Enrich single-code directions ---
    # Related codes: if direction has only 1 filter, add its natural companions
    _RELATED_CODES = {
        "StPO": ["BStKR", "JStPO"],
        "StGB": ["JStG"],
        "OR": ["ZGB"],
        "ZGB": ["OR"],
        "BGG": ["BV"],
        "BV": ["BGG"],
        "IVG": ["ATSG"],
        "ATSG": ["IVG", "IVV"],
        "AIG": ["BV"],
        "DBG": ["StHG"],
        "StHG": ["DBG"],
        "1B_": ["7B_"],
        "7B_": ["1B_"],
        "6B_": ["BGE_IV"],
        "BGE_IV": ["6B_"],
        "8C_": ["9C_", "BGE_V"],
        "9C_": ["8C_", "BGE_V"],
        "BGE_V": ["8C_", "9C_"],
        "4A_": ["4D_", "BGE_III"],
        "5A_": ["BGE_III"],
        "2C_": ["BGE_I", "BGE_II"],
    }
    
    # Rechtsgebiet -> fallback codes when dedup strips everything
    _DOMAIN_FALLBACK = {
        "strafprozess": ["StPO", "BStKR", "JStPO"],
        "strafrecht": ["StGB", "JStG"],
        "zivilrecht": ["OR", "ZGB"],
        "familienrecht": ["ZGB", "OR"],
        "prozessrecht": ["BGG", "BV"],
        "verfahrensrecht": ["BGG", "BV", "VwVG"],
        "sozialversicherung": ["IVG", "ATSG", "IVV"],
        "iv": ["IVG", "ATSG"],
        "oeffentliches_recht": ["AIG", "BV", "VwVG"],
        "steuerrecht": ["DBG", "StHG"],
        "finanzmarktrecht": ["FIDLEG", "FINMAG"],
        "strafverfahren": ["StPO", "BStKR"],
        "leitentscheide": ["BGE_I", "BGE_II", "BGE_III", "BGE_IV", "BGE_V"],
    }

    used_codes = set()
    for direction in directions:
        # Deduplicate: remove codes already used by higher-priority directions
        original = direction.filter_codes[:]
        direction.filter_codes = [c for c in direction.filter_codes if c not in used_codes]
        
        # Enrich: if 0 or 1 codes remain after dedup
        if len(direction.filter_codes) == 0:
            # All codes stripped -> assign from rechtsgebiet
            rg = direction.rechtsgebiet.lower().replace("-", "").replace(" ", "_")
            # Try matching domain keys
            fallback = []
            for key, codes in _DOMAIN_FALLBACK.items():
                if key in rg or rg in key:
                    fallback = codes
                    break
            # Add unused fallback codes
            for fc in fallback:
                if fc not in used_codes:
                    if fc in law_code_to_indices or fc in court_code_to_indices:
                        direction.filter_codes.append(fc)
                if len(direction.filter_codes) >= 3:
                    break
        
        if len(direction.filter_codes) == 1:
            # Single code -> add related companions
            base = direction.filter_codes[0]
            for related in _RELATED_CODES.get(base, []):
                if related not in used_codes and related not in direction.filter_codes:
                    if related in law_code_to_indices or related in court_code_to_indices:
                        direction.filter_codes.append(related)
        
        # Track all codes this direction now owns
        used_codes.update(direction.filter_codes)
    # --- END POST-PROCESSING ---

    plan = Plan(
        sachverhalt=data.get("sachverhalt", ""),
        rechtsfragen=data.get("rechtsfragen", []),
        directions=sorted(directions, key=lambda d: d.priority),
    )
    L.debug(f"\n  PLAN SUMMARY: {len(plan.directions)} directions total")
    L.debug("")
    return plan


def run_direction_logged(
    question: str,
    plan: Plan,
    direction: Direction,
    direction_idx: int,
    prior_findings: list[tuple[str, float, str]],
) -> list[tuple[str, float, str]]:
    """Instrumented run_direction with full logging."""
    _log_header(f"DIRECTION {direction_idx+1}/{len(plan.directions)}", 2)
    L.debug(f"  Priority: {direction.priority}")
    L.debug(f"  Corpus: {direction.corpus}")
    L.debug(f"  Rechtsgebiet: {direction.rechtsgebiet}")
    L.debug(f"  Filter codes: {direction.filter_codes}")
    L.debug(f"  Reasoning: {direction.reasoning}")
    L.debug(f"  Seed queries: {direction.seed_queries}")
    L.debug(f"  Prior findings count: {len(prior_findings)}")
    L.debug("")
    
    start_time = time.time()
    direction_citations: list[tuple[str, float, str]] = []
    history: list[dict] = []

    # --- Iteration 0: Seed query ---
    if direction.seed_queries:
        seed_q = direction.seed_queries[0]
        L.debug(f"  [Iter 0] SEED QUERY (no LLM): \"{seed_q}\"")
        
        # Log filter pool size
        if direction.filter_codes:
            pool_sizes = []
            code_map = law_code_to_indices if direction.corpus == "laws" else court_code_to_indices
            for fc in direction.filter_codes:
                if fc in code_map:
                    pool_sizes.append(f"{fc}={len(code_map[fc])}")
            L.debug(f"    Filter pool: {', '.join(pool_sizes)} docs in scope")
        else:
            L.debug(f"    Filter: NONE (searching full corpus)")
        
        results = filtered_hybrid_search(seed_q, direction.corpus, direction.filter_codes,
                                          top_k=CONFIG["search_top_k"])
        direction_citations.extend(results)
        history.append({"query": seed_q, "results": results})
        
        L.debug(f"    Results: {len(results)} hits")
        for j, (cit, score, snippet) in enumerate(results[:10]):
            L.debug(f"      {j+1}. {cit} (score={score:.3f}) â€” {snippet[:80]}...")
        if len(results) > 10:
            L.debug(f"      ... +{len(results)-10} more")
        L.debug("")

    # --- ReAct iterations ---
    is_procedural = direction.priority >= 90
    plan_summary = (
        f"Sachverhalt: {plan.sachverhalt}\n"
        f"Rechtsfragen: {'; '.join(plan.rechtsfragen)}\n"
        f"Dies ist Richtung {direction_idx+1} von {len(plan.directions)}."
    )

    for iteration in range(1, CONFIG["max_executor_iterations"] + 1):
        if time.time() - start_time > CONFIG["executor_timeout_sec"]:
            L.debug(f"  [Iter {iteration}] TIMEOUT â€” stopping direction")
            break

        L.debug(f"  [Iter {iteration}] LLM CALL (executor)")
        
        # Build prompt
        if is_procedural:
            system_content = executor_procedural_template.format(
                prior_findings=format_prior_findings(prior_findings + direction_citations),
            )
            L.debug(f"    Template: PROCEDURAL")
        else:
            taxonomy_section = get_taxonomy_section(direction.filter_codes)
            system_content = executor_system_template.format(
                rechtsgebiet=direction.rechtsgebiet,
                corpus=direction.corpus,
                filter_codes=", ".join(direction.filter_codes),
                reasoning=direction.reasoning,
                taxonomy_section=taxonomy_section,
                plan_summary=plan_summary,
                prior_findings=format_prior_findings(prior_findings + direction_citations),
                direction_history=format_direction_history(history),
            )
            L.debug(f"    Template: STANDARD | Taxonomy: {len(taxonomy_section)} chars")
        
        L.debug(f"    Prompt length: {len(system_content):,} chars")

        prompt = f"[INST] {system_content}\n\nGeneriere deine nÃ¤chste Suchanfrage oder signalisiere done. [/INST]"
        t0 = time.time()
        response = llm(
            prompt,
            max_tokens=CONFIG["max_tokens_executor"],
            temperature=CONFIG.get("temperature_executor", 0.3),
            grammar=executor_grammar,
            stop=["[INST]", "</s>"],
        )
        elapsed = time.time() - t0
        raw = response["choices"][0]["text"].strip()
        
        L.debug(f"    LLM time: {elapsed:.2f}s")
        L.debug(f"    Raw output: {raw}")

        # Parse
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            L.debug(f"    âœ— JSON parse failed â€” stopping direction")
            break

        # Check done
        if parsed.get("done", False) or not parsed.get("query", "").strip():
            L.debug(f"    â†’ DONE signal received (done={parsed.get('done')}, query='{parsed.get('query', '')}')")
            if parsed.get("reasoning"):
                L.debug(f"    â†’ Reasoning: {parsed['reasoning']}")
            break

        # Check for repeated query (exact match)
        query = parsed["query"].strip()
        if query in [h["query"] for h in history]:
            break

        # --- EMBEDDING DIVERSITY CHECK ---
        # Reject queries that are semantically too similar to previous ones
        if history:
            new_emb = embed_query(query, direction.corpus)  # (1, dim)
            too_similar = False
            for prev in history:
                prev_emb = embed_query(prev["query"], direction.corpus)
                sim = float((new_emb @ prev_emb.T)[0, 0])
                if sim > 0.80:
                    too_similar = True
                    break
            if too_similar:
                break  # Query too similar to a previous one

        # Execute search
        results = filtered_hybrid_search(query, direction.corpus, direction.filter_codes,
                                          top_k=CONFIG["search_top_k"])

        # Adaptive fallback
        if not results and direction.filter_codes:
            L.debug(f"    â†’ 0 results with filter {direction.filter_codes} â€” trying UNFILTERED fallback")
            results = filtered_hybrid_search(query, direction.corpus, [], top_k=CONFIG["search_top_k"])
            L.debug(f"    â†’ Unfiltered fallback returned: {len(results)} results")

        direction_citations.extend(results)
        history.append({"query": query, "results": results})
        
        L.debug(f"    Results: {len(results)} hits")
        for j, (cit, score, snippet) in enumerate(results[:10]):
            L.debug(f"      {j+1}. {cit} (score={score:.3f}) â€” {snippet[:80]}...")
        if len(results) > 10:
            L.debug(f"      ... +{len(results)-10} more")
        L.debug("")

    elapsed_total = time.time() - start_time
    L.debug(f"\n  Direction {direction_idx+1} COMPLETE: {len(direction_citations)} citations in {elapsed_total:.1f}s")
    
    # Deduplicated summary
    unique_cits = list(set(c for c, _, *_rest in direction_citations))
    L.debug(f"  Unique citations: {len(unique_cits)}")
    for c in unique_cits[:15]:
        L.debug(f"    â€¢ {c}")
    if len(unique_cits) > 15:
        L.debug(f"    ... +{len(unique_cits)-15} more")
    L.debug("")
    
    return direction_citations


def run_pipeline_logged(question: str, gold: str = "", query_idx: int = 0) -> str:
    """Fully instrumented pipeline with comprehensive logging."""
    t_start = time.time()
    
    _log_separator("â•")
    L.debug(f"  QUERY {query_idx+1}")
    _log_separator("â•")
    L.debug(f"  Question: {question}")
    L.debug(f"  Gold citations: {gold}")
    if gold:
        gold_list = gold.split(";")
        L.debug(f"  Gold count: {len(gold_list)}")
        for g in gold_list[:20]:
            L.debug(f"    â€¢ {g.strip()}")
        if len(gold_list) > 20:
            L.debug(f"    ... +{len(gold_list)-20} more")
    L.debug(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    L.debug("")

    # â•â•â• PHASE 1: PLANNER â•â•â•
    plan = run_planner_logged(question)
    if plan is None:
        L.debug("  âš  Planner returned None â†’ using FALLBACK decomposition")
        plan = fallback_decompose(question)
        L.debug(f"  Fallback plan: {len(plan.directions)} directions")
        for i, d in enumerate(plan.directions):
            L.debug(f"    Dir {i+1}: corpus={d.corpus}, codes={d.filter_codes}, reason={d.reasoning}")
        L.debug("")

    # Validate: ensure min 3 directions
    while len(plan.directions) < 3:
        L.debug("  âš  Plan has <3 directions â€” adding catch-all broad search")
        plan.directions.insert(0, Direction(
            priority=50, corpus="laws" if len(plan.directions) % 2 == 0 else "courts",
            rechtsgebiet="Allgemein",
            filter_codes=[], reasoning="Catch-all broad search",
            seed_queries=[plan.sachverhalt[:80]] if plan.sachverhalt else [],
        ))

    # â•â•â• PHASE 2: DIRECTION EXECUTORS â•â•â•
    _log_header("PHASE 2: DIRECTION EXECUTORS", 1)
    L.debug(f"  Executing {len(plan.directions)} directions...\n")

    all_direction_citations: list[list[tuple[str, float, str]]] = []
    prior_findings: list[tuple[str, float, str]] = []

    for i, direction in enumerate(plan.directions):
        direction_cits = run_direction_logged(question, plan, direction, i, prior_findings)
        all_direction_citations.append(direction_cits)
        prior_findings = (prior_findings + direction_cits)[-20:]

    # â•â•â• PHASE 3: AGGREGATION â•â•â•
    _log_header("PHASE 3: AGGREGATION + RERANKING", 1)
    
    # Flatten all citations with provenance tracking
    all_cits_flat = []
    citation_provenance: dict[str, list[int]] = {}  # cit â†’ [direction indices]
    for dir_idx, direction_cits in enumerate(all_direction_citations):
        all_cits_flat.extend(direction_cits)
        for cit, _, *_rest in direction_cits:
            citation_provenance.setdefault(cit, []).append(dir_idx + 1)
    
    L.debug(f"  Total raw citations (all directions): {len(all_cits_flat)}")
    
    # Unique pre-aggregation
    unique_pre = set(c for c, _, *_rest in all_cits_flat)
    L.debug(f"  Unique citations before aggregation: {len(unique_pre)}")
    L.debug(f"  Duplicates removed: {len(all_cits_flat) - len(unique_pre)}")
    
    # Per-direction contribution
    _log_header("Per-Direction Contribution", 3)
    for dir_idx, direction_cits in enumerate(all_direction_citations):
        unique_in_dir = set(c for c, _, *_rest in direction_cits)
        L.debug(f"    Direction {dir_idx+1}: {len(direction_cits)} raw, {len(unique_in_dir)} unique")
    
    # Log what reranker will receive as its query
    _actual_rerank_q = plan.sachverhalt if plan.sachverhalt else question
    _log_header("RERANKER INPUT QUERY", 2)
    L.debug(f"  sachverhalt present: {bool(plan.sachverhalt)}")
    L.debug(f"  Using: " + ("plan.sachverhalt" if plan.sachverhalt else "question (fallback â€” sachverhalt was empty)"))
    L.debug(f"  Rerank query text:")
    L.debug(f"    {_actual_rerank_q}")
    L.debug("")

    # Run aggregation WITH diagnostics (single reranker pass)
    result, diag = aggregate_and_output(all_direction_citations, question, rerank_query=plan.sachverhalt, return_diagnostics=True)
    
    # Log diagnostics from aggregation
    L.debug(f"\n  Detected case type: {diag['case_type']}")
    L.debug(f"  Detected prefixes: {diag['prefixes']}")
    L.debug(f"  Procedural defaults injected: {len(diag['defaults'])}")
    for d in diag['defaults']:
        L.debug(f"    + {d}")
    
    # â”€â”€â”€ Reranking details â”€â”€â”€
    _log_header("RERANKER (Qwen3-Reranker-0.6B)", 2)
    L.debug(f"  Stage: Phase 3 â€” after all directions complete, reranking combined unique citations")
    L.debug(f"  Input: {len(diag['candidates'])} candidates (unique + defaults)")
    L.debug(f"  Cutoff threshold: {CONFIG['rerank_score_cutoff']}")
    L.debug(f"  Max output: {CONFIG['max_final_citations']}")
    L.debug(f"  Reranker time: {diag['rerank_time']:.2f}s")
    
    scored_all = diag['scored_all']
    if scored_all:
        L.debug(f"\n  ALL RERANKER SCORES (sorted desc):")
        L.debug(f"  {'Rank':<5} {'Score':<8} {'RRF':<6} {'Dirs':<10} {'Citation':<40} {'Text Preview'}")
        L.debug(f"  {'-'*5} {'-'*8} {'-'*6} {'-'*10} {'-'*40} {'-'*40}")
        
        cutoff = CONFIG["rerank_score_cutoff"]
        n_above = 0
        n_below = 0
        for rank, (cit, rscore) in enumerate(scored_all, 1):
            rrf_score = diag['citation_scores_rrf'].get(cit, 0.0)
            dirs = citation_provenance.get(cit, ["default"])
            above = rscore >= cutoff
            marker = "âœ“" if above else "âœ—"
            if above:
                n_above += 1
            else:
                n_below += 1
            # Log all above cutoff + first 10 below cutoff
            if above or n_below <= 10:
                preview = citation_to_text.get(cit, cit)[:50].replace('\n', ' ')
                L.debug(f"  {marker} {rank:<4} {rscore:<8.4f} {rrf_score:<6.3f} {str(dirs):<10} {cit:<40} {preview}")
        
        if n_below > 10:
            L.debug(f"  ... +{n_below - 10} more below cutoff (not shown)")
        
        L.debug(f"\n  RERANKER SUMMARY:")
        L.debug(f"    Above cutoff ({cutoff}): {n_above}")
        L.debug(f"    Below cutoff (dropped): {n_below}")
        L.debug(f"    Score range: {scored_all[0][1]:.4f} â†’ {scored_all[-1][1]:.4f}")
        if n_above > 0:
            above_scores = [s for _, s in scored_all if s >= cutoff]
            L.debug(f"    Kept scores: mean={np.mean(above_scores):.4f}, min={min(above_scores):.4f}")
        if n_below > 0:
            below_scores = [s for _, s in scored_all if s < cutoff]
            L.debug(f"    Dropped scores: max={max(below_scores):.4f} (closest to cutoff)")
    
    # Citations that used fallback text
    if diag['no_text']:
        L.debug(f"\n  âš  CITATIONS WITH NO DOC TEXT (reranked on citation string only): {len(diag['no_text'])}")
        for c in diag['no_text'][:10]:
            L.debug(f"    - {c}")
    
    # Explicit citations from question
    if diag['explicit']:
        L.debug(f"\n  Explicit citations extracted from question text: {diag['explicit']}")
    
    # Log final output
    _log_header("FINAL OUTPUT", 1)
    final_cits = result.split(";") if result else []
    L.debug(f"  Final citation count: {len(final_cits)}")
    for i, cit in enumerate(final_cits):
        dirs = citation_provenance.get(cit, ["default/explicit"])
        L.debug(f"    {i+1}. {cit}  [from direction(s): {dirs}]")
    
    # Compare with gold
    if gold:
        _log_header("EVALUATION vs GOLD", 2)
        pred_set = set(final_cits)
        gold_set = set(g.strip() for g in gold.split(";") if g.strip())
        
        tp = pred_set & gold_set
        fp = pred_set - gold_set
        fn = gold_set - pred_set
        
        precision = len(tp) / len(pred_set) if pred_set else 0.0
        recall = len(tp) / len(gold_set) if gold_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        L.debug(f"  Precision: {precision:.4f} ({len(tp)}/{len(pred_set)})")
        L.debug(f"  Recall:    {recall:.4f} ({len(tp)}/{len(gold_set)})")
        L.debug(f"  F1:        {f1:.4f}")
        L.debug(f"\n  TRUE POSITIVES ({len(tp)}):")
        for c in sorted(tp):
            L.debug(f"    âœ“ {c}")
        L.debug(f"\n  FALSE POSITIVES ({len(fp)}) â€” predicted but not in gold:")
        for c in sorted(fp):
            L.debug(f"    âœ— {c}")
        L.debug(f"\n  FALSE NEGATIVES ({len(fn)}) â€” in gold but missed:")
        for c in sorted(fn):
            L.debug(f"    âœ— {c}")
    
    elapsed = time.time() - t_start
    L.debug(f"\n  Total pipeline time: {elapsed:.1f}s")
    _log_separator("â•")
    L.debug("\n\n")
    
    return result


print(f"Pipeline logger ready â†’ {LOG_PATH}")
print(f"  All pipeline output will be logged to this file.")
print(f"  Use run_pipeline_logged() for full instrumentation.")

Pipeline logger ready â†’ /kaggle/working/pipeline_debug_log.txt
  All pipeline output will be logged to this file.
  Use run_pipeline_logged() for full instrumentation.


In [ ]:
# Cell 17: Test on Validation Set (with full logging)
# Every step is logged to OUTPUT_PATH / "pipeline_debug_log.txt"

def compute_f1(predicted: str, gold: str) -> tuple[float, float, float]:
    """Compute citation-level F1."""
    pred_set = set(predicted.split(";")) if predicted else set()
    gold_set = set(gold.split(";")) if gold else set()
    if not pred_set and not gold_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not gold_set:
        return 0.0, 0.0, 0.0
    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set) if gold_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


# Log session header
L.debug(f"{'#'*80}")
L.debug(f"  PIPELINE VALIDATION RUN")
L.debug(f"  Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
L.debug(f"  Config: {CONFIG}")
L.debug(f"{'#'*80}\n\n")

# Try val.csv or train.csv with gold citations
val_path = DATA_PATH / "val.csv"
if not val_path.exists():
    val_path = TRAIN_CSV

val_df = pd.read_csv(val_path)
has_gold = "gold_citations" in val_df.columns

if has_gold:
    print(f"Running validation on {len(val_df)} queries...")
    print(f"Full log: {LOG_PATH}\n")
    
    L.debug(f"Validation dataset: {val_path}")
    L.debug(f"Total queries: {len(val_df)}\n\n")
    
    results = []
    MAX_VAL_QUERIES = 1  # DEBUG: limit to first N queries
    for idx, row in val_df.head(MAX_VAL_QUERIES).iterrows():
        question = row["query"] if "query" in row.index else row.get("query_text", row.get("question", ""))
        gold = str(row["gold_citations"]) if pd.notna(row.get("gold_citations")) else ""

        predicted = run_pipeline_logged(question, gold=gold, query_idx=idx)
        p, r, f1 = compute_f1(predicted, gold)
        results.append({"query_id": row.get("query_id", idx), "P": p, "R": r, "F1": f1})

        print(f"  [{idx+1}/{len(val_df)}] P={p:.3f} R={r:.3f} F1={f1:.3f} â€” {question[:60]}...")

        # Persist log every 5 queries (survives kernel crash)
        if (idx + 1) % 5 == 0:
            _file_handler.flush()
            _save_checkpoint(LOG_PATH)

    results_df = pd.DataFrame(results)
    
    # Log summary
    _log_separator("#")
    L.debug("  VALIDATION SUMMARY")
    _log_separator("#")
    L.debug(f"  Queries: {len(results_df)}")
    L.debug(f"  Macro Precision: {results_df['P'].mean():.4f}")
    L.debug(f"  Macro Recall:    {results_df['R'].mean():.4f}")
    L.debug(f"  Macro F1:        {results_df['F1'].mean():.4f}")
    L.debug(f"\n  Per-query breakdown:")
    for _, r in results_df.iterrows():
        L.debug(f"    {r['query_id']}: P={r['P']:.3f} R={r['R']:.3f} F1={r['F1']:.3f}")
    
    print(f"\n{'='*50}")
    print(f"MACRO AVERAGES:")
    print(f"  Precision: {results_df['P'].mean():.4f}")
    print(f"  Recall:    {results_df['R'].mean():.4f}")
    print(f"  F1:        {results_df['F1'].mean():.4f}")
    print(f"\nFull debug log saved: {LOG_PATH}")
    _file_handler.flush()
    _save_checkpoint(LOG_PATH)
else:
    print(f"No gold_citations column in {val_path.name}. Running single test...")
    test_q = "Under what conditions can pre-trial detention be extended beyond the initial period?"
    
    predicted = run_pipeline_logged(test_q, gold="", query_idx=0)
    print(f"\nResult ({len(predicted.split(';'))} citations): {predicted[:200]}...")
    print(f"\nFull debug log saved: {LOG_PATH}")
    _file_handler.flush()
    _save_checkpoint(LOG_PATH)

Running validation on 10 queries...
Full log: /kaggle/working/pipeline_debug_log.txt

  [1/10] P=0.033 R=0.048 F1=0.039 â€” May a court lawfully order a three‑month extension of pre‑tr...

MACRO AVERAGES:
  Precision: 0.0333
  Recall:    0.0476
  F1:        0.0392

Full debug log saved: /kaggle/working/pipeline_debug_log.txt
âœ… SAVED to kaggle.com/datasets/charan1996/rag-checkpoints-v2 (push #1): faiss_laws_qwen3_embeddings.pkl, faiss_courts_qwen3_embeddings.pkl, corpus_documents.pkl, pipeline_debug_log.txt, planner_system.txt, executor_system.txt, executor_procedural.txt, planner.gbnf, executor.gbnf, fallback_rules.txt, swiss_legal_system.txt, routing_guide_laws.txt, routing_guide_courts.txt, terminology_bridge.txt, procedural_defaults.txt


In [ ]:
# Cell 18: Generate Test Predictions

test_df = pd.read_csv(TEST_CSV)
print(f"Generating predictions for {len(test_df)} test queries...\n")

predictions = []
for idx, row in test_df.iterrows():
    question = row["query"] if "query" in row.index else str(row.get("query_text", row.get("question", "")))
    query_id = row.get("query_id", idx)

    t0 = time.time()
    predicted = run_pipeline(question, verbose=False)
    elapsed = time.time() - t0

    predictions.append({"query_id": query_id, "predicted_citations": predicted})
    print(f"  [{idx+1}/{len(test_df)}] {elapsed:.1f}s â€” {len(predicted.split(';'))} citations â€” {question[:60]}...")

    # Checkpoint every 10 â€” save to persistent dataset
    if (idx + 1) % 10 == 0:
        _prog_path = OUTPUT_PATH / "submission_progress.csv"
        pd.DataFrame(predictions).to_csv(_prog_path, index=False)
        _save_checkpoint(_prog_path)

print(f"\nAll {len(predictions)} predictions generated.")

In [21]:
# Cell 19: Save Submission

submission_df = pd.DataFrame(predictions)
submission_path = OUTPUT_PATH / "submission.csv"
submission_df.to_csv(submission_path, index=False)

# Push final submission to persistent dataset
_save_checkpoint(submission_path)

print(f"Submission saved: {submission_path}")
print(f"  Rows: {len(submission_df)}")
print(f"  Columns: {list(submission_df.columns)}")
print(f"  âœ… Pushed to kaggle.com/datasets/{DATASET_SLUG}")

print(f"\nFirst 3 rows:")
for _, row in submission_df.head(3).iterrows():
    cits = str(row['predicted_citations']).split(';')
    print(f"  {row['query_id']}: {len(cits)} citations â€” {';'.join(cits[:3])}...")